# Master Model Evaluation - Official Test Set

**Purpose:** Evaluate all trained models on the official held-out test set  
**Test Set:** 968 images (patient-stratified, never seen during training)  
**Models:** ResNet-50 Baseline, SE-ResNet50, CBAM-ResNet50, ViT-B/16

## Evaluation Protocol

1. Discover all trained model checkpoints
2. Load best checkpoint for each model/seed combination
3. Evaluate on official test set
4. Compute comprehensive metrics (accuracy, F1, precision, recall, per-class)
5. Generate comparison tables and statistics

In [1]:
# IMPORTS
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from pathlib import Path
import numpy as np
import pandas as pd
import re
from collections import defaultdict
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Imports successful")

Imports successful


In [2]:
# CONFIGURATION

ROOT = Path(r"D:\Keel\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training")

CHECKPOINT_DIR = ROOT / "Checkpoints"
TEST_PATH = ROOT / "Data_Kermany_OCT2017" / "test"

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
CLASS_NAMES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*80)
print("CONFIGURATION")
print("="*80)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Test set path: {TEST_PATH}")
print(f"Device: {DEVICE}")
print("="*80)

CONFIGURATION
Checkpoint directory: D:\Keel\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints
Test set path: D:\Keel\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Data_Kermany_OCT2017\test
Device: cuda


In [3]:
# MODEL ARCHITECTURE DEFINITIONS

class SEBlock(nn.Module):
    """Squeeze-and-Excitation block for channel attention."""
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class SEResNet50(nn.Module):
    """ResNet-50 with Squeeze-and-Excitation attention."""
    def __init__(self, num_classes=4, pretrained=False, reduction=16):
        super(SEResNet50, self).__init__()
        resnet = models.resnet50(pretrained=pretrained)
        
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        
        self.se1 = SEBlock(256, reduction)
        self.se2 = SEBlock(512, reduction)
        self.se3 = SEBlock(1024, reduction)
        self.se4 = SEBlock(2048, reduction)
        
        self.avgpool = resnet.avgpool
        self.fc = nn.Linear(2048, num_classes)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.se1(x)
        x = self.layer2(x)
        x = self.se2(x)
        x = self.layer3(x)
        x = self.se3(x)
        x = self.layer4(x)
        x = self.se4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


class ChannelAttention(nn.Module):
    """Channel attention for CBAM."""
    def __init__(self, channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.mlp(self.avg_pool(x).view(b, c))
        max_out = self.mlp(self.max_pool(x).view(b, c))
        out = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * out.expand_as(x)


class SpatialAttention(nn.Module):
    """Spatial attention for CBAM."""
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.sigmoid(self.conv(out))
        return x * out


class CBAM(nn.Module):
    """Convolutional Block Attention Module."""
    def __init__(self, channels, reduction=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)
    
    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


class CBAMResNet50(nn.Module):
    """ResNet-50 with CBAM attention."""
    def __init__(self, num_classes=4, pretrained=False, reduction=16):
        super(CBAMResNet50, self).__init__()
        resnet = models.resnet50(pretrained=pretrained)
        
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        
        self.cbam1 = CBAM(256, reduction)
        self.cbam2 = CBAM(512, reduction)
        self.cbam3 = CBAM(1024, reduction)
        self.cbam4 = CBAM(2048, reduction)
        
        self.avgpool = resnet.avgpool
        self.fc = nn.Linear(2048, num_classes)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.cbam1(x)
        x = self.layer2(x)
        x = self.cbam2(x)
        x = self.layer3(x)
        x = self.cbam3(x)
        x = self.layer4(x)
        x = self.cbam4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


def create_model(model_type, num_classes=4):
    """Factory function to create models by type."""
    if model_type == 'resnet_baseline':
        model = models.resnet50(pretrained=False)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_type == 'se_resnet':
        model = SEResNet50(num_classes=num_classes, pretrained=False)
    elif model_type == 'cbam_resnet':
        model = CBAMResNet50(num_classes=num_classes, pretrained=False)
    elif model_type == 'vit_b16':
        model = vit_b_16(weights=None)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    else:
        raise ValueError(f"Unknown model type: {model_type}")
    
    return model


print("Model architectures defined")

Model architectures defined


In [4]:
# CHECKPOINT DISCOVERY

def discover_checkpoints(checkpoint_dir, verbose=False):
    """
    Discover all model checkpoints and organize by serial/seed.
    
    Returns dict: {serial_number: {seed: {model_name, checkpoints}}}
    """
    checkpoint_dir = Path(checkpoint_dir)
    
    # Pattern: SERIAL_modelname_seedN_epochM_mode_timestamp.pth
    pattern = re.compile(r'^(\d+)_([a-z0-9_]+)_seed(\d+)_epoch(\d+)_(best|last|intermediate)_.*\.pth$')
    
    discovered = defaultdict(lambda: defaultdict(lambda: {'checkpoints': [], 'model_name': None}))
    
    checkpoint_files = list(checkpoint_dir.glob("*.pth"))
    
    if verbose:
        print(f"\nScanning {len(checkpoint_files)} checkpoint files...\n")
    
    for filepath in checkpoint_files:
        match = pattern.match(filepath.name)
        if match:
            serial = int(match.group(1))
            model_name = match.group(2)
            seed = int(match.group(3))
            epoch = int(match.group(4))
            mode = match.group(5)
            
            discovered[serial][seed]['model_name'] = model_name
            discovered[serial][seed]['checkpoints'].append({
                'path': filepath,
                'epoch': epoch,
                'mode': mode,
                'filename': filepath.name
            })
            
            if verbose:
                print(f"  Serial {serial:02d}, Seed {seed}, {model_name}, Epoch {epoch}, {mode}")
    
    # Convert to regular dict for cleaner output
    return {k: dict(v) for k, v in discovered.items()}


def find_best_checkpoint(checkpoints_dict, serial, seed):
    """
    Find the best checkpoint for a given serial/seed combination.
    Priority: best mode > last mode > highest epoch intermediate
    """
    if serial not in checkpoints_dict or seed not in checkpoints_dict[serial]:
        return None
    
    checkpoints = checkpoints_dict[serial][seed]['checkpoints']
    
    # Priority 1: best checkpoint
    best_ckpts = [c for c in checkpoints if c['mode'] == 'best']
    if best_ckpts:
        return max(best_ckpts, key=lambda x: x['epoch'])
    
    # Priority 2: last checkpoint
    last_ckpts = [c for c in checkpoints if c['mode'] == 'last']
    if last_ckpts:
        return max(last_ckpts, key=lambda x: x['epoch'])
    
    # Priority 3: highest epoch intermediate
    if checkpoints:
        return max(checkpoints, key=lambda x: x['epoch'])
    
    return None


# Discover all checkpoints
print("="*80)
print("DISCOVERING CHECKPOINTS")
print("="*80)

all_checkpoints = discover_checkpoints(CHECKPOINT_DIR, verbose=True)

print("\n" + "="*80)
print("SUMMARY")
print("="*80)

for serial in sorted(all_checkpoints.keys()):
    for seed in sorted(all_checkpoints[serial].keys()):
        info = all_checkpoints[serial][seed]
        model_name = info['model_name']
        num_checkpoints = len(info['checkpoints'])
        best = find_best_checkpoint(all_checkpoints, serial, seed)
        
        print(f"Serial {serial:02d} | Seed {seed} | {model_name:20s} | "
              f"{num_checkpoints:2d} checkpoints | Best: Epoch {best['epoch'] if best else 'N/A'}")

print("="*80)

DISCOVERING CHECKPOINTS

Scanning 511 checkpoint files...

  Serial 01, Seed 42, resnet_baseline, Epoch 10, intermediate
  Serial 01, Seed 42, resnet_baseline, Epoch 11, best
  Serial 01, Seed 42, resnet_baseline, Epoch 13, best
  Serial 01, Seed 42, resnet_baseline, Epoch 14, best
  Serial 01, Seed 42, resnet_baseline, Epoch 15, intermediate
  Serial 01, Seed 42, resnet_baseline, Epoch 1, best
  Serial 01, Seed 42, resnet_baseline, Epoch 20, best
  Serial 01, Seed 42, resnet_baseline, Epoch 20, intermediate
  Serial 01, Seed 42, resnet_baseline, Epoch 21, best
  Serial 01, Seed 42, resnet_baseline, Epoch 25, intermediate
  Serial 01, Seed 42, resnet_baseline, Epoch 27, best
  Serial 01, Seed 42, resnet_baseline, Epoch 28, best
  Serial 01, Seed 42, resnet_baseline, Epoch 2, best
  Serial 01, Seed 42, resnet_baseline, Epoch 30, intermediate
  Serial 01, Seed 42, resnet_baseline, Epoch 35, intermediate
  Serial 01, Seed 42, resnet_baseline, Epoch 38, best
  Serial 01, Seed 42, resnet_ba

In [5]:
# TEST SET LOADING

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = ImageFolder(root=str(TEST_PATH), transform=test_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

print("="*80)
print("TEST SET LOADED")
print("="*80)
print(f"Total images: {len(test_dataset)}")
print(f"Batches: {len(test_loader)}")
print(f"Classes: {test_dataset.classes}")
print("="*80)

TEST SET LOADED
Total images: 968
Batches: 31
Classes: ['CNV', 'DME', 'DRUSEN', 'NORMAL']


In [6]:
# EVALUATION FUNCTION — RETAINS SAMPLE-LEVEL OUTPUTS

def evaluate_model(model, test_loader, device):
    """
    Evaluate one model and retain everything required for the classification
    safety audit. Probabilities and uncertainty scores are calculated from the
    same inference pass, so no second model run is needed.
    """
    model.eval()
    all_logits = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating", leave=False):
            images = images.to(device, non_blocking=True)
            logits = model(images)
            predicted = torch.argmax(logits, dim=1)

            all_logits.append(logits.detach().cpu())
            all_preds.append(predicted.detach().cpu())
            all_labels.append(labels.detach().cpu())

    logits = torch.cat(all_logits, dim=0).numpy()
    all_preds = torch.cat(all_preds, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy()

    # Stable softmax and output-space audit scores.
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    probabilities = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    sorted_probs = np.sort(probabilities, axis=1)
    confidence = sorted_probs[:, -1]
    margin = sorted_probs[:, -1] - sorted_probs[:, -2]
    entropy = -(probabilities * np.log(np.clip(probabilities, 1e-12, 1.0))).sum(axis=1)
    normalized_entropy = entropy / np.log(NUM_CLASSES)
    # Standard free-energy convention with temperature T=1.
    energy = -(np.log(np.exp(shifted).sum(axis=1)) + logits.max(axis=1))

    accuracy = accuracy_score(all_labels, all_preds) * 100
    f1_macro = f1_score(all_labels, all_preds, average='macro') * 100
    precision_macro = precision_score(
        all_labels, all_preds, average='macro', zero_division=0
    ) * 100
    recall_macro = recall_score(
        all_labels, all_preds, average='macro', zero_division=0
    ) * 100

    f1_per_class = f1_score(
        all_labels, all_preds, labels=range(NUM_CLASSES),
        average=None, zero_division=0
    ) * 100
    precision_per_class = precision_score(
        all_labels, all_preds, labels=range(NUM_CLASSES),
        average=None, zero_division=0
    ) * 100
    recall_per_class = recall_score(
        all_labels, all_preds, labels=range(NUM_CLASSES),
        average=None, zero_division=0
    ) * 100
    cm = confusion_matrix(all_labels, all_preds, labels=range(NUM_CLASSES))

    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_per_class': f1_per_class,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'confusion_matrix': cm,
        'predictions': all_preds,
        'labels': all_labels,
        'logits': logits,
        'probabilities': probabilities,
        'confidence': confidence,
        'entropy': entropy,
        'normalized_entropy': normalized_entropy,
        'margin': margin,
        'energy': energy
    }


print("Evaluation function defined with sample-level audit outputs")


Evaluation function defined with sample-level audit outputs


In [7]:
# EVALUATE ALL MODELS

print("="*80)
print("EVALUATING ALL MODELS")
print("="*80)

results = []

for serial in sorted(all_checkpoints.keys()):
    for seed in sorted(all_checkpoints[serial].keys()):
        info = all_checkpoints[serial][seed]
        model_name = info['model_name']
        
        best_checkpoint = find_best_checkpoint(all_checkpoints, serial, seed)
        
        if best_checkpoint is None:
            print(f"\n⚠️  No checkpoint found for Serial {serial:02d}, Seed {seed}")
            continue
        
        print(f"\n{'='*80}")
        print(f"Serial {serial:02d} | Seed {seed} | {model_name.upper()}")
        print(f"Checkpoint: {best_checkpoint['filename']}")
        print(f"Epoch: {best_checkpoint['epoch']} | Mode: {best_checkpoint['mode']}")
        print(f"{'='*80}")
        
        # Create model
        model = create_model(model_name, NUM_CLASSES)
        
        # Load checkpoint
        checkpoint = torch.load(best_checkpoint['path'], map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model = model.to(DEVICE)
        
        # Evaluate
        metrics = evaluate_model(model, test_loader, DEVICE)
        
        # Store results
        result = {
            'serial': serial,
            'seed': seed,
            'model_name': model_name,
            'epoch': best_checkpoint['epoch'],
            'checkpoint_mode': best_checkpoint['mode'],
            'checkpoint_filename': best_checkpoint['filename'],
            **metrics
        }
        results.append(result)
        
        # Print results
        print(f"\nTest Set Results:")
        print(f"  Accuracy:  {metrics['accuracy']:.2f}%")
        print(f"  F1-Score:  {metrics['f1_macro']:.2f}%")
        print(f"  Precision: {metrics['precision_macro']:.2f}%")
        print(f"  Recall:    {metrics['recall_macro']:.2f}%")
        
        print(f"\nPer-Class F1-Scores:")
        for i, class_name in enumerate(CLASS_NAMES):
            print(f"  {class_name:8s}: {metrics['f1_per_class'][i]:.2f}%")

print("\n" + "="*80)
print("EVALUATION COMPLETE")
print("="*80)

EVALUATING ALL MODELS

Serial 01 | Seed 42 | RESNET_BASELINE
Checkpoint: 01_resnet_baseline_seed42_epoch46_best_20260114_001728.pth
Epoch: 46 | Mode: best



Test Set Results:
  Accuracy:  99.07%
  F1-Score:  99.07%
  Precision: 99.09%
  Recall:    99.07%

Per-Class F1-Scores:
  CNV     : 98.37%
  DME     : 99.79%
  DRUSEN  : 98.33%
  NORMAL  : 99.79%

Serial 02 | Seed 42 | SE_RESNET
Checkpoint: 02_se_resnet_seed42_epoch47_best_20260114_020922.pth
Epoch: 47 | Mode: best



Test Set Results:
  Accuracy:  98.24%
  F1-Score:  98.25%
  Precision: 98.34%
  Recall:    98.24%

Per-Class F1-Scores:
  CNV     : 96.80%
  DME     : 99.79%
  DRUSEN  : 96.60%
  NORMAL  : 99.79%

Serial 03 | Seed 84 | RESNET_BASELINE
Checkpoint: 03_resnet_baseline_seed84_epoch50_best_20260114_101157.pth
Epoch: 50 | Mode: best



Test Set Results:
  Accuracy:  98.86%
  F1-Score:  98.87%
  Precision: 98.90%
  Recall:    98.86%

Per-Class F1-Scores:
  CNV     : 97.98%
  DME     : 99.79%
  DRUSEN  : 97.90%
  NORMAL  : 99.79%

Serial 04 | Seed 42 | SE_RESNET
Checkpoint: 04_se_resnet_seed42_epoch47_best_20260114_115805.pth
Epoch: 47 | Mode: best



Test Set Results:
  Accuracy:  98.24%
  F1-Score:  98.25%
  Precision: 98.34%
  Recall:    98.24%

Per-Class F1-Scores:
  CNV     : 96.80%
  DME     : 99.79%
  DRUSEN  : 96.60%
  NORMAL  : 99.79%

Serial 05 | Seed 84 | SE_RESNET
Checkpoint: 05_se_resnet_seed84_epoch42_best_20260114_134347.pth
Epoch: 42 | Mode: best


...


Test Set Results:
  Accuracy:  98.35%
  F1-Score:  98.35%
  Precision: 98.43%
  Recall:    98.35%

Per-Class F1-Scores:
  CNV     : 96.79%
  DME     : 99.79%
  DRUSEN  : 96.80%
  NORMAL  : 100.00%

Serial 15 | Seed 126 | VIT_B16
Checkpoint: 15_vit_b16_seed126_epoch50_best_20260116_170850.pth
Epoch: 50 | Mode: best



Test Set Results:
  Accuracy:  98.55%
  F1-Score:  98.56%
  Precision: 98.63%
  Recall:    98.55%

Per-Class F1-Scores:
  CNV     : 97.19%
  DME     : 99.59%
  DRUSEN  : 97.46%
  NORMAL  : 100.00%

Serial 16 | Seed 3407 | VIT_B16
Checkpoint: 16_vit_b16_seed3407_epoch39_best_20260116_205856.pth
Epoch: 39 | Mode: best



Test Set Results:
  Accuracy:  98.24%
  F1-Score:  98.24%
  Precision: 98.36%
  Recall:    98.24%

Per-Class F1-Scores:
  CNV     : 96.61%
  DME     : 100.00%
  DRUSEN  : 96.36%
  NORMAL  : 100.00%

Serial 17 | Seed 3407 | RESNET_BASELINE
Checkpoint: 17_resnet_baseline_seed3407_epoch42_best_20260116_234021.pth
Epoch: 42 | Mode: best



Test Set Results:
  Accuracy:  98.76%
  F1-Score:  98.76%
  Precision: 98.78%
  Recall:    98.76%

Per-Class F1-Scores:
  CNV     : 97.97%
  DME     : 99.59%
  DRUSEN  : 97.91%
  NORMAL  : 99.59%

Serial 18 | Seed 3407 | CBAM_RESNET
Checkpoint: 18_cbam_resnet_seed3407_epoch40_best_20260117_020155.pth
Epoch: 40 | Mode: best



Test Set Results:
  Accuracy:  97.83%
  F1-Score:  97.84%
  Precision: 98.00%
  Recall:    97.83%

Per-Class F1-Scores:
  CNV     : 95.84%
  DME     : 98.96%
  DRUSEN  : 96.58%
  NORMAL  : 100.00%

EVALUATION COMPLETE


In [8]:
# RESULTS SUMMARY TABLE

print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)

# Create summary DataFrame
summary_data = []
for r in results:
    summary_data.append({
        'Serial': r['serial'],
        'Seed': r['seed'],
        'Model': r['model_name'],
        'Epoch': r['epoch'],
        'Accuracy': f"{r['accuracy']:.2f}%",
        'F1': f"{r['f1_macro']:.2f}%",
        'Precision': f"{r['precision_macro']:.2f}%",
        'Recall': f"{r['recall_macro']:.2f}%"
    })

summary_df = pd.DataFrame(summary_data)
print("\n", summary_df.to_string(index=False))

print("\n" + "="*80)


RESULTS SUMMARY

  Serial  Seed           Model  Epoch Accuracy     F1 Precision Recall
      1    42 resnet_baseline     46   99.07% 99.07%    99.09% 99.07%
      2    42       se_resnet     47   98.24% 98.25%    98.34% 98.24%
      3    84 resnet_baseline     50   98.86% 98.87%    98.90% 98.86%
      4    42       se_resnet     47   98.24% 98.25%    98.34% 98.24%
      5    84       se_resnet     42   98.45% 98.45%    98.52% 98.45%
      6   126       se_resnet     48   98.35% 98.35%    98.42% 98.35%
      7    42     cbam_resnet     49   97.93% 97.94%    98.02% 97.93%
      8    84     cbam_resnet     33   98.55% 98.56%    98.62% 98.55%
      9   126     cbam_resnet     45   98.76% 98.76%    98.79% 98.76%
     10   126 resnet_baseline     39   99.07% 99.07%    99.08% 99.07%
     11  3407       se_resnet     42   98.04% 98.04%    98.18% 98.04%
     12    42         vit_b16     48   98.97% 98.97%    99.01% 98.97%
     13    84         vit_b16      4   97.42% 97.42%    97.51% 97.42%
 

In [9]:
# FILTER INCOMPLETE RUNS AND COMPUTE CORRECTED STATISTICS

import numpy as np
import pandas as pd

# Define results directory
results_dir = CHECKPOINT_DIR / "evaluation_results"
results_dir.mkdir(exist_ok=True)

print("\n" + "="*80)
print("FILTERING INCOMPLETE TRAINING RUNS")
print("="*80)

# Create DataFrame directly from results (not summary_data)
df_all = pd.DataFrame([{
    'Serial': r['serial'],
    'Seed': r['seed'],
    'Model': r['model_name'],
    'Epoch': r['epoch'],
    'Accuracy': r['accuracy'],  # Already numeric
    'F1': r['f1_macro'],  # Already numeric
    'Precision': r['precision_macro'],  # Already numeric
    'Recall': r['recall_macro']  # Already numeric
} for r in results])

# Identify incomplete runs (epoch < 30 suggests interrupted training)
INCOMPLETE_EPOCH_THRESHOLD = 30

incomplete_runs = df_all[df_all['Epoch'] < INCOMPLETE_EPOCH_THRESHOLD]
complete_runs = df_all[df_all['Epoch'] >= INCOMPLETE_EPOCH_THRESHOLD]

if len(incomplete_runs) > 0:
    print(f"\n⚠️  INCOMPLETE RUNS DETECTED (Epoch < {INCOMPLETE_EPOCH_THRESHOLD}):")
    print(incomplete_runs.to_string(index=False))
    print(f"\n✓ These {len(incomplete_runs)} run(s) will be EXCLUDED from statistics")
else:
    print(f"\n✓ No incomplete runs detected (all epochs >= {INCOMPLETE_EPOCH_THRESHOLD})")

print(f"\n✓ Complete runs: {len(complete_runs)}")
print("="*80)

# Data is already numeric - no conversion needed!
df_stats = complete_runs.copy()

# Compute statistics
summary = (
    df_stats.groupby("Model")
    .agg(
        Total_Runs=("Serial", "count"),
        Unique_Seeds=("Seed", pd.Series.nunique),
        Acc_Mean=("Accuracy", "mean"),
        Acc_Std=("Accuracy", "std"),
        Acc_Min=("Accuracy", "min"),
        Acc_Max=("Accuracy", "max"),
        F1_Mean=("F1", "mean"),
        F1_Std=("F1", "std")
    )
    .reset_index()
)

# Sort by Acc_Max BEFORE formatting (while still numeric)
summary = summary.sort_values(by="Acc_Max", ascending=False).reset_index(drop=True)

# Format to percentages AFTER sorting
def fmt_pct(x):
    return f"{x:.2f}%" if not np.isnan(x) else "N/A"

pct_cols = ["Acc_Mean", "Acc_Std", "Acc_Min", "Acc_Max", "F1_Mean", "F1_Std"]
for col in pct_cols:
    summary[col] = summary[col].apply(fmt_pct)

# Display corrected statistics
print("\n" + "="*80)
print("CORRECTED MULTI-SEED STATISTICS")
print("="*80)
print("(Incomplete runs excluded)")
print("\n" + summary.to_string(index=False))
print("\n" + "="*80)

# Save corrected statistics
corrected_csv = results_dir / "corrected_multi_seed_statistics.csv"
summary.to_csv(corrected_csv, index=False)
print(f"\n✅ Corrected statistics saved: {corrected_csv}")
print("="*80)



FILTERING INCOMPLETE TRAINING RUNS

⚠️  INCOMPLETE RUNS DETECTED (Epoch < 30):
 Serial  Seed   Model  Epoch  Accuracy        F1  Precision    Recall
     13    84 vit_b16      4 97.417355 97.417474   97.50562 97.417355

✓ These 1 run(s) will be EXCLUDED from statistics

✓ Complete runs: 17

CORRECTED MULTI-SEED STATISTICS
(Incomplete runs excluded)

          Model  Total_Runs  Unique_Seeds Acc_Mean Acc_Std Acc_Min Acc_Max F1_Mean F1_Std
resnet_baseline           4             4   98.94%   0.15%  98.76%  99.07%  98.94%  0.15%
        vit_b16           4             4   98.53%   0.32%  98.24%  98.97%  98.53%  0.32%
    cbam_resnet           4             4   98.27%   0.46%  97.83%  98.76%  98.28%  0.45%
      se_resnet           5             4   98.26%   0.15%  98.04%  98.45%  98.27%  0.15%


✅ Corrected statistics saved: D:\Keel\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints\evaluation_results\corrected_multi_seed_statistics.csv


In [10]:
# GENERATE CONFUSION MATRICES AND ROC CURVES

from PIL import Image, ImageDraw, ImageFont
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import io

print("\n" + "="*80)
print("GENERATING VISUALIZATIONS")
print("="*80)

# Create visualizations directory
viz_dir = results_dir / "visualizations"
viz_dir.mkdir(exist_ok=True)

def create_confusion_matrix_pil(cm, class_names, title, save_path):
    """
    Create confusion matrix visualization using PIL.
    
    Args:
        cm: Confusion matrix (numpy array)
        class_names: List of class names
        title: Title for the plot
        save_path: Path to save the image
    """
    n_classes = len(class_names)
    
    # Image dimensions
    cell_size = 100
    margin = 150
    width = margin + cell_size * n_classes + 100
    height = margin + cell_size * n_classes + 50
    
    # Create image
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)
    
    # Try to use a font, fall back to default if not available
    try:
        title_font = ImageFont.truetype("arial.ttf", 20)
        label_font = ImageFont.truetype("arial.ttf", 14)
        cell_font = ImageFont.truetype("arial.ttf", 16)
    except:
        title_font = ImageFont.load_default()
        label_font = ImageFont.load_default()
        cell_font = ImageFont.load_default()
    
    # Draw title
    draw.text((width // 2 - 100, 20), title, fill='black', font=title_font)
    
    # Normalize confusion matrix for color mapping
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Draw confusion matrix cells
    for i in range(n_classes):
        for j in range(n_classes):
            # Cell position
            x = margin + j * cell_size
            y = margin + i * cell_size
            
            # Cell color (green for correct, red for incorrect)
            if i == j:
                # Correct predictions - shades of green
                intensity = int(255 * (1 - cm_normalized[i, j]))
                color = (intensity, 255, intensity)
            else:
                # Incorrect predictions - shades of red
                if cm[i, j] > 0:
                    intensity = int(255 * (1 - min(cm_normalized[i, j] * 10, 1)))
                    color = (255, intensity, intensity)
                else:
                    color = (255, 255, 255)
            
            # Draw cell
            draw.rectangle([x, y, x + cell_size, y + cell_size], 
                          fill=color, outline='black', width=2)
            
            # Draw count
            count_text = str(cm[i, j])
            bbox = draw.textbbox((0, 0), count_text, font=cell_font)
            text_width = bbox[2] - bbox[0]
            text_height = bbox[3] - bbox[1]
            text_x = x + (cell_size - text_width) // 2
            text_y = y + (cell_size - text_height) // 2
            draw.text((text_x, text_y), count_text, fill='black', font=cell_font)
    
    # Draw row labels (True labels)
    draw.text((10, margin + n_classes * cell_size // 2), 
             "True", fill='black', font=label_font)
    for i, class_name in enumerate(class_names):
        y = margin + i * cell_size + cell_size // 2
        draw.text((margin - 80, y), class_name, fill='black', font=label_font)
    
    # Draw column labels (Predicted labels)
    draw.text((margin + n_classes * cell_size // 2, height - 30), 
             "Predicted", fill='black', font=label_font)
    for j, class_name in enumerate(class_names):
        x = margin + j * cell_size + 10
        draw.text((x, margin - 30), class_name, fill='black', font=label_font)
    
    # Save image
    img.save(save_path)
    print(f"  ✓ Saved: {save_path.name}")
    
    return img


def create_roc_curve_pil(y_true, y_pred_proba, class_names, title, save_path):
    """
    Create ROC curve visualization using PIL.
    
    Args:
        y_true: True labels
        y_pred_proba: Predicted probabilities (n_samples, n_classes)
        class_names: List of class names
        title: Title for the plot
        save_path: Path to save the image
    """
    n_classes = len(class_names)
    
    # Image dimensions
    width, height = 800, 600
    margin_left, margin_right = 80, 50
    margin_top, margin_bottom = 80, 80
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    
    # Create image
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)
    
    try:
        title_font = ImageFont.truetype("arial.ttf", 18)
        label_font = ImageFont.truetype("arial.ttf", 12)
    except:
        title_font = ImageFont.load_default()
        label_font = ImageFont.load_default()
    
    # Draw title
    draw.text((width // 2 - 100, 20), title, fill='black', font=title_font)
    
    # Draw axes
    draw.line([(margin_left, margin_top), 
              (margin_left, height - margin_bottom)], fill='black', width=2)
    draw.line([(margin_left, height - margin_bottom), 
              (width - margin_right, height - margin_bottom)], fill='black', width=2)
    
    # Draw diagonal (random classifier)
    draw.line([(margin_left, height - margin_bottom),
              (width - margin_right, margin_top)], fill='gray', width=1)
    
    # Binarize labels for one-vs-rest ROC
    y_true_bin = label_binarize(y_true, classes=range(n_classes))
    
    # Colors for each class
    colors = ['red', 'blue', 'green', 'orange']
    
    # Compute and draw ROC curve for each class
    for i, (class_name, color) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        
        # Convert to pixel coordinates
        points = []
        for fp, tp in zip(fpr, tpr):
            x = margin_left + int(fp * plot_width)
            y = height - margin_bottom - int(tp * plot_height)
            points.append((x, y))
        
        # Draw ROC curve
        if len(points) > 1:
            draw.line(points, fill=color, width=2)
        
        # Draw legend
        legend_y = margin_top + 20 + i * 20
        draw.line([(width - margin_right - 150, legend_y),
                  (width - margin_right - 130, legend_y)], fill=color, width=2)
        draw.text((width - margin_right - 120, legend_y - 10),
                 f"{class_name}: {roc_auc:.3f}", fill='black', font=label_font)
    
    # Draw axis labels
    draw.text((width // 2 - 60, height - 30), 
             "False Positive Rate", fill='black', font=label_font)
    draw.text((10, height // 2 - 10), 
             "True Positive Rate", fill='black', font=label_font)
    
    # Draw ticks
    for i in range(11):
        # X-axis ticks
        x = margin_left + int(i * plot_width / 10)
        draw.line([(x, height - margin_bottom), 
                  (x, height - margin_bottom + 5)], fill='black', width=1)
        draw.text((x - 10, height - margin_bottom + 10), 
                 f"{i/10:.1f}", fill='black', font=label_font)
        
        # Y-axis ticks
        y = height - margin_bottom - int(i * plot_height / 10)
        draw.line([(margin_left - 5, y), 
                  (margin_left, y)], fill='black', width=1)
        draw.text((margin_left - 30, y - 10), 
                 f"{i/10:.1f}", fill='black', font=label_font)
    
    # Save image
    img.save(save_path)
    print(f"  ✓ Saved: {save_path.name}")
    
    return img


# Generate visualizations for each complete model
print(f"\nGenerating confusion matrices and ROC curves...")
print(f"Output directory: {viz_dir}")
print()

for r in results:
    # Skip incomplete runs
    if r['epoch'] < INCOMPLETE_EPOCH_THRESHOLD:
        print(f"⊘ Skipping Serial {r['serial']:02d} (incomplete, epoch {r['epoch']})")
        continue
    
    serial = r['serial']
    seed = r['seed']
    model_name = r['model_name']
    
    print(f"\nSerial {serial:02d} | {model_name} | Seed {seed}")
    
    # Get confusion matrix
    cm = r['confusion_matrix']
    y_true = r['labels']
    y_pred = r['predictions']
    
    # Generate confusion matrix image
    cm_title = f"Serial {serial:02d} - {model_name.replace('_', ' ').title()} (Seed {seed})"
    cm_path = viz_dir / f"{serial:02d}_{model_name}_seed{seed}_confusion_matrix.png"
    create_confusion_matrix_pil(cm, CLASS_NAMES, cm_title, cm_path)
    
    # Reuse probabilities retained during the main evaluation pass.
    all_probs = r['probabilities']
    
    # Generate ROC curve image
    roc_title = f"Serial {serial:02d} - {model_name.replace('_', ' ').title()} (Seed {seed})"
    roc_path = viz_dir / f"{serial:02d}_{model_name}_seed{seed}_roc_curve.png"
    create_roc_curve_pil(y_true, all_probs, CLASS_NAMES, roc_title, roc_path)

print("\n" + "="*80)
print("VISUALIZATION COMPLETE")
print("="*80)
print(f"✓ Confusion matrices saved to: {viz_dir}")
print(f"✓ ROC curves saved to: {viz_dir}")
print("="*80)




GENERATING VISUALIZATIONS

Generating confusion matrices and ROC curves...
Output directory: D:\Keel\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints\evaluation_results\visualizations


Serial 01 | resnet_baseline | Seed 42
  ✓ Saved: 01_resnet_baseline_seed42_confusion_matrix.png
  ✓ Saved: 01_resnet_baseline_seed42_roc_curve.png

Serial 02 | se_resnet | Seed 42
  ✓ Saved: 02_se_resnet_seed42_confusion_matrix.png
  ✓ Saved: 02_se_resnet_seed42_roc_curve.png

Serial 03 | resnet_baseline | Seed 84
  ✓ Saved: 03_resnet_baseline_seed84_confusion_matrix.png
  ✓ Saved: 03_resnet_baseline_seed84_roc_curve.png

Serial 04 | se_resnet | Seed 42
  ✓ Saved: 04_se_resnet_seed42_confusion_matrix.png
  ✓ Saved: 04_se_resnet_seed42_roc_curve.png

Serial 05 | se_resnet | Seed 84
  ✓ Saved: 05_se_resnet_seed84_confusion_matrix.png
  ✓ Saved: 05_se_resnet_seed84_roc_curve.png

Serial 06 | se_resnet | Seed 126
  ✓ Saved: 06_se_resnet_seed126_confusion_matrix.png
  ✓ Saved: 06

In [11]:
# ENHANCED SUMMARY WITH ACCURACY SORTING (INCOMPLETE RUNS EXCLUDED)

import numpy as np
import pandas as pd

# Define results directory
results_dir = CHECKPOINT_DIR / "evaluation_results"
results_dir.mkdir(exist_ok=True)



# Print legend
print("\n" + "-"*80)
print("METRIC ABBREVIATIONS LEGEND:")
print("-"*80)
print("  Acc       : Accuracy (overall classification accuracy)")
print("  F1        : F1-Score (macro-averaged across all classes)")
print("  Prec      : Precision (macro-averaged)")
print("  Rec       : Recall (macro-averaged)")
print("  Epoch     : Training epoch number")
print("  Seed      : Random seed used for training")
print("  Serial    : Sequential checkpoint number")
print("  Mode      : Checkpoint mode from evaluation")
print("-"*80 + "\n")


print("\n" + "="*80)
print("ENHANCED STATISTICS (Sorted by Best Accuracy)")
print("="*80)
print("(Incomplete runs excluded)")
print()

if not results:
    print("No results to analyze")
else:
    # Build DataFrame from results
    summary_data = []
    for r in results:
        summary_data.append({
            'Serial': r['serial'],
            'Seed': r['seed'],
            'Model': r['model_name'],
            'Epoch': r['epoch'],
            'Accuracy': r['accuracy'],
            'F1': r['f1_macro'],
            'Precision': r['precision_macro'],
            'Recall': r['recall_macro']
        })
    
    df_all = pd.DataFrame(summary_data)
    
    # Filter out incomplete runs (same threshold as before)
    INCOMPLETE_EPOCH_THRESHOLD = 30
    df_stats = df_all[df_all['Epoch'] >= INCOMPLETE_EPOCH_THRESHOLD].copy()
    
    n_excluded = len(df_all) - len(df_stats)
    if n_excluded > 0:
        excluded_serials = df_all[df_all['Epoch'] < INCOMPLETE_EPOCH_THRESHOLD]['Serial'].tolist()
        print(f"⚠️  {n_excluded} incomplete run(s) excluded: Serial {excluded_serials}\n")
    
    # Group by model and compute statistics
    summary = (
        df_stats.groupby("Model")
        .agg(
            Runs=("Serial", "count"),
            Seeds=("Seed", pd.Series.nunique),
            Acc_Mean=("Accuracy", "mean"),
            Acc_Std=("Accuracy", "std"),
            Acc_Min=("Accuracy", "min"),
            Acc_Max=("Accuracy", "max"),
            F1_Mean=("F1", "mean"),
            F1_Std=("F1", "std")
        )
        .reset_index()
    )
    
    # Sort by best accuracy (while still numeric)
    summary = summary.sort_values(by="Acc_Max", ascending=False).reset_index(drop=True)
    
    # Format to percentages AFTER sorting
    def fmt_pct(x):
        return f"{x:.2f}%" if not np.isnan(x) else "N/A"
    
    pct_cols = ["Acc_Mean", "Acc_Std", "Acc_Min", "Acc_Max", "F1_Mean", "F1_Std"]
    for col in pct_cols:
        summary[col] = summary[col].apply(fmt_pct)
    
    # Display
    print(summary.to_string(index=False))
    print("\n" + "="*80)
    
    # Save to CSV
    csv_path = results_dir / "enhanced_statistics.csv"
    summary.to_csv(csv_path, index=False)
    print(f"\n✅ Enhanced statistics saved: {csv_path}")
    print("="*80)


--------------------------------------------------------------------------------
METRIC ABBREVIATIONS LEGEND:
--------------------------------------------------------------------------------
  Acc       : Accuracy (overall classification accuracy)
  F1        : F1-Score (macro-averaged across all classes)
  Prec      : Precision (macro-averaged)
  Rec       : Recall (macro-averaged)
  Epoch     : Training epoch number
  Seed      : Random seed used for training
  Serial    : Sequential checkpoint number
  Mode      : Checkpoint mode from evaluation
--------------------------------------------------------------------------------


ENHANCED STATISTICS (Sorted by Best Accuracy)
(Incomplete runs excluded)

⚠️  1 incomplete run(s) excluded: Serial [13]

          Model  Runs  Seeds Acc_Mean Acc_Std Acc_Min Acc_Max F1_Mean F1_Std
resnet_baseline     4      4   98.94%   0.15%  98.76%  99.07%  98.94%  0.15%
        vit_b16     4      4   98.53%   0.32%  98.24%  98.97%  98.53%  0.32%
    cbam_

In [12]:
# INDIVIDUAL MODEL RANKINGS - BEST MODEL AT TOP

import numpy as np
import pandas as pd

print("\n" + "="*80)
print("🏆 INDIVIDUAL MODEL RANKINGS (SORTED BY ACCURACY - BEST FIRST)")
print("="*80)
print("All individual runs ranked by test set accuracy")
print()

if not results:
    print("No results to analyze")
else:
    # Build detailed DataFrame
    individual_data = []
    for r in results:
        individual_data.append({
            'Model': r['model_name'],
            'Serial': r['serial'],
            'Seed': r['seed'],
            'Accuracy': r['accuracy'],
            'F1': r['f1_macro'],
            'Precision': r['precision_macro'],
            'Recall': r['recall_macro'],
            'Epoch': r['epoch']
        })
    
    df_individual = pd.DataFrame(individual_data)
    
    # Filter out incomplete runs (same threshold as before)
    INCOMPLETE_EPOCH_THRESHOLD = 30
    df_complete = df_individual[df_individual['Epoch'] >= INCOMPLETE_EPOCH_THRESHOLD].copy()
    
    n_excluded = len(df_individual) - len(df_complete)
    if n_excluded > 0:
        excluded_serials = df_individual[df_individual['Epoch'] < INCOMPLETE_EPOCH_THRESHOLD]['Serial'].tolist()
        print(f"⚠️  {n_excluded} incomplete run(s) excluded: Serial {excluded_serials}\n")
    
    # Sort by accuracy (best first)
    df_ranked = df_complete.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
    
    # Add rank column
    df_ranked.insert(0, 'Rank', range(1, len(df_ranked) + 1))
    
    # Format percentages
    def fmt_pct(x):
        return f"{x:.2f}%"
    
    # Create display dataframe with formatted values
    df_display = df_ranked.copy()
    df_display['Accuracy'] = df_display['Accuracy'].apply(fmt_pct)
    df_display['F1'] = df_display['F1'].apply(fmt_pct)
    df_display['Precision'] = df_display['Precision'].apply(fmt_pct)
    df_display['Recall'] = df_display['Recall'].apply(fmt_pct)
    
    # Display full ranking table
    print(df_display.to_string(index=False))
    
    # Highlight best model
    best = df_ranked.iloc[0]
    print("\n" + "="*80)
    print("🥇 BEST MODEL (Highest Test Set Accuracy)")
    print("="*80)
    print(f"  Model:      {best['Model']}")
    print(f"  Serial:     {best['Serial']:02d}")
    print(f"  Seed:       {best['Seed']}")
    print(f"  Accuracy:   {best['Accuracy']:.2f}%")
    print(f"  F1-Score:   {best['F1']:.2f}%")
    print(f"  Precision:  {best['Precision']:.2f}%")
    print(f"  Recall:     {best['Recall']:.2f}%")
    print(f"  Epoch:      {best['Epoch']}")
    print("="*80)
    
    # Show top 5
    print("\n" + "="*80)
    print("🏅 TOP 5 MODELS")
    print("="*80)
    top5 = df_display.head(5)[['Rank', 'Model', 'Serial', 'Seed', 'Accuracy', 'F1']]
    print(top5.to_string(index=False))
    print("="*80)
    
    # Save individual rankings to CSV
    results_dir = CHECKPOINT_DIR / "evaluation_results"
    individual_csv = results_dir / "individual_model_rankings.csv"
    
    # Save with numeric values for further analysis
    df_ranked.to_csv(individual_csv, index=False)
    print(f"\n✅ Individual rankings saved: {individual_csv}")
    print("="*80)


🏆 INDIVIDUAL MODEL RANKINGS (SORTED BY ACCURACY - BEST FIRST)
All individual runs ranked by test set accuracy

⚠️  1 incomplete run(s) excluded: Serial [13]

 Rank           Model  Serial  Seed Accuracy     F1 Precision Recall  Epoch
    1 resnet_baseline       1    42   99.07% 99.07%    99.09% 99.07%     46
    2 resnet_baseline      10   126   99.07% 99.07%    99.08% 99.07%     39
    3         vit_b16      12    42   98.97% 98.97%    99.01% 98.97%     48
    4 resnet_baseline       3    84   98.86% 98.87%    98.90% 98.86%     50
    5 resnet_baseline      17  3407   98.76% 98.76%    98.78% 98.76%     42
    6     cbam_resnet       9   126   98.76% 98.76%    98.79% 98.76%     45
    7     cbam_resnet       8    84   98.55% 98.56%    98.62% 98.55%     33
    8         vit_b16      15   126   98.55% 98.56%    98.63% 98.55%     50
    9       se_resnet       5    84   98.45% 98.45%    98.52% 98.45%     42
   10       se_resnet       6   126   98.35% 98.35%    98.42% 98.35%     48
   11

In [13]:
# SAVE RESULTS

results_dir = CHECKPOINT_DIR / "evaluation_results"
results_dir.mkdir(exist_ok=True)

# MULTI-SEED STATISTICS

print("\n" + "="*80)
print("MULTI-SEED STATISTICS")
print("="*80)

# Group by model type
model_stats = defaultdict(lambda: {'accuracy': [], 'f1': []})

for r in results:
    model_stats[r['model_name']]['accuracy'].append(r['accuracy'])
    model_stats[r['model_name']]['f1'].append(r['f1_macro'])

stats_data = []
for model_name in sorted(model_stats.keys()):
    acc_values = model_stats[model_name]['accuracy']
    f1_values = model_stats[model_name]['f1']
    
    if len(acc_values) > 1:
        stats_data.append({
            'Model': model_name,
            'Seeds': len(acc_values),
            'Acc Mean': f"{np.mean(acc_values):.2f}%",
            'Acc Std': f"{np.std(acc_values):.2f}%",
            'F1 Mean': f"{np.mean(f1_values):.2f}%",
            'F1 Std': f"{np.std(f1_values):.2f}%"
        })
    else:
        stats_data.append({
            'Model': model_name,
            'Seeds': len(acc_values),
            'Acc Mean': f"{acc_values[0]:.2f}%",
            'Acc Std': 'N/A',
            'F1 Mean': f"{f1_values[0]:.2f}%",
            'F1 Std': 'N/A'
        })

if stats_data:
    stats_df = pd.DataFrame(stats_data)
    print("\n", stats_df.to_string(index=False))


print("\n" + "="*80)

# Save summary
summary_df.to_csv(results_dir / "test_results_summary.csv", index=False)
print(f"\n✅ Summary saved: {results_dir / 'test_results_summary.csv'}")

# Save statistics
if stats_data:
    stats_df.to_csv(results_dir / "multi_seed_statistics.csv", index=False)
    print(f"✅ Statistics saved: {results_dir / 'multi_seed_statistics.csv'}")

print("\n" + "="*80)
print("EVALUATION COMPLETE - ALL RESULTS SAVED")
print("="*80)


MULTI-SEED STATISTICS

           Model  Seeds Acc Mean Acc Std F1 Mean F1 Std
    cbam_resnet      4   98.27%   0.40%  98.28%  0.39%
resnet_baseline      4   98.94%   0.13%  98.94%  0.13%
      se_resnet      5   98.26%   0.14%  98.27%  0.14%
        vit_b16      5   98.31%   0.51%  98.31%  0.51%


✅ Summary saved: D:\Keel\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints\evaluation_results\test_results_summary.csv
✅ Statistics saved: D:\Keel\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints\evaluation_results\multi_seed_statistics.csv

EVALUATION COMPLETE - ALL RESULTS SAVED


In [14]:
# COMPLETE CLASSIFICATION-AUDIT EXPORTS
# Run after all model evaluations. This cell does not rerun inference.

from sklearn.metrics import precision_recall_fscore_support

audit_dir = results_dir / "classification_audit"
audit_dir.mkdir(parents=True, exist_ok=True)

NORMAL_INDEX = CLASS_NAMES.index("NORMAL")
INCOMPLETE_EPOCH_THRESHOLD = 30

sample_paths = [str(Path(path).resolve()) for path, _ in test_dataset.samples]
relative_paths = [
    Path(path).resolve().relative_to(TEST_PATH.resolve()).as_posix()
    for path in sample_paths
]

prediction_frames = []
run_rows = []
cm_rows = []
per_class_rows = []

for r in results:
    labels = np.asarray(r["labels"], dtype=int)
    preds = np.asarray(r["predictions"], dtype=int)
    probs = np.asarray(r["probabilities"], dtype=float)
    logits = np.asarray(r["logits"], dtype=float)

    if not (len(labels) == len(preds) == len(sample_paths) == len(probs)):
        raise ValueError(
            f"Sample count mismatch for serial {r['serial']}, seed {r['seed']}"
        )

    is_complete = r["epoch"] >= INCOMPLETE_EPOCH_THRESHOLD
    normal_decision = preds == NORMAL_INDEX
    safe_normal = normal_decision & (labels == NORMAL_INDEX)
    false_normal = normal_decision & (labels != NORMAL_INDEX)

    frame = pd.DataFrame({
        "model": r["model_name"],
        "serial": r["serial"],
        "seed": r["seed"],
        "epoch": r["epoch"],
        "checkpoint_mode": r["checkpoint_mode"],
        "checkpoint_filename": r.get("checkpoint_filename", ""),
        "complete_run": is_complete,
        "sample_index": np.arange(len(labels)),
        "image_id": relative_paths,
        "image_path": sample_paths,
        "true_label_index": labels,
        "true_label": [CLASS_NAMES[i] for i in labels],
        "predicted_label_index": preds,
        "predicted_label": [CLASS_NAMES[i] for i in preds],
        "prob_CNV": probs[:, CLASS_NAMES.index("CNV")],
        "prob_DME": probs[:, CLASS_NAMES.index("DME")],
        "prob_DRUSEN": probs[:, CLASS_NAMES.index("DRUSEN")],
        "prob_NORMAL": probs[:, NORMAL_INDEX],
        "logit_CNV": logits[:, CLASS_NAMES.index("CNV")],
        "logit_DME": logits[:, CLASS_NAMES.index("DME")],
        "logit_DRUSEN": logits[:, CLASS_NAMES.index("DRUSEN")],
        "logit_NORMAL": logits[:, NORMAL_INDEX],
        "confidence": r["confidence"],
        "entropy": r["entropy"],
        "normalized_entropy": r["normalized_entropy"],
        "classification_margin": r["margin"],
        "energy_T1": r["energy"],
        "is_correct": labels == preds,
        "predicted_NORMAL": normal_decision,
        "safe_NORMAL": safe_normal,
        "false_NORMAL": false_normal,
    })
    prediction_frames.append(frame)

    run_rows.append({
        "model": r["model_name"],
        "serial": r["serial"],
        "seed": r["seed"],
        "epoch": r["epoch"],
        "checkpoint_mode": r["checkpoint_mode"],
        "checkpoint_filename": r.get("checkpoint_filename", ""),
        "complete_run": is_complete,
        "test_images": len(labels),
        "accuracy_pct": r["accuracy"],
        "macro_precision_pct": r["precision_macro"],
        "macro_recall_pct": r["recall_macro"],
        "macro_f1_pct": r["f1_macro"],
        "predicted_NORMAL": int(normal_decision.sum()),
        "safe_NORMAL": int(safe_normal.sum()),
        "false_NORMAL": int(false_normal.sum()),
        "NORMAL_false_negative_rate_pct": (
            100.0 * false_normal.sum() / (labels != NORMAL_INDEX).sum()
        ),
    })

    cm = np.asarray(r["confusion_matrix"], dtype=int)
    for true_i, true_name in enumerate(CLASS_NAMES):
        for pred_i, pred_name in enumerate(CLASS_NAMES):
            cm_rows.append({
                "model": r["model_name"],
                "serial": r["serial"],
                "seed": r["seed"],
                "epoch": r["epoch"],
                "complete_run": is_complete,
                "true_label": true_name,
                "predicted_label": pred_name,
                "count": int(cm[true_i, pred_i]),
            })

    class_precision, class_recall, class_f1, class_support = (
        precision_recall_fscore_support(
            labels, preds, labels=range(NUM_CLASSES), zero_division=0
        )
    )
    for class_i, class_name in enumerate(CLASS_NAMES):
        per_class_rows.append({
            "model": r["model_name"],
            "serial": r["serial"],
            "seed": r["seed"],
            "epoch": r["epoch"],
            "complete_run": is_complete,
            "class": class_name,
            "precision_pct": 100 * class_precision[class_i],
            "recall_pct": 100 * class_recall[class_i],
            "f1_pct": 100 * class_f1[class_i],
            "support": int(class_support[class_i]),
        })

all_predictions_df = pd.concat(prediction_frames, ignore_index=True)
run_summary_df = pd.DataFrame(run_rows).sort_values(["serial", "seed"])
confusion_counts_df = pd.DataFrame(cm_rows)
per_class_metrics_df = pd.DataFrame(per_class_rows)

# Main exports
all_predictions_df.to_csv(audit_dir / "sample_level_predictions_all_runs.csv", index=False)
run_summary_df.to_csv(audit_dir / "normal_decision_summary_all_runs.csv", index=False)
confusion_counts_df.to_csv(audit_dir / "confusion_matrix_counts_all_runs.csv", index=False)
per_class_metrics_df.to_csv(audit_dir / "per_class_metrics_all_runs.csv", index=False)

# Focused exports for direct inspection and manuscript tables.
complete_predictions_df = all_predictions_df[all_predictions_df["complete_run"]].copy()
false_normal_df = complete_predictions_df[complete_predictions_df["false_NORMAL"]].copy()
safe_normal_df = complete_predictions_df[complete_predictions_df["safe_NORMAL"]].copy()
complete_predictions_df.to_csv(
    audit_dir / "sample_level_predictions_complete_runs.csv", index=False
)
false_normal_df.to_csv(
    audit_dir / "false_NORMAL_cases_complete_runs.csv", index=False
)
safe_normal_df.to_csv(
    audit_dir / "safe_NORMAL_cases_complete_runs.csv", index=False
)

# One row per image shows whether the same scan is missed by multiple runs.
false_normal_overlap = (
    false_normal_df.groupby(["image_id", "true_label"], as_index=False)
    .agg(
        missed_by_run_count=("false_NORMAL", "size"),
        missed_by_model_count=("model", "nunique"),
        models=("model", lambda x: "|".join(sorted(set(x)))),
        run_keys=("serial", lambda x: "|".join(map(str, sorted(set(x))))),
        max_prob_NORMAL=("prob_NORMAL", "max"),
        min_prob_NORMAL=("prob_NORMAL", "min"),
    )
)
false_normal_overlap.to_csv(
    audit_dir / "false_NORMAL_unique_image_overlap.csv", index=False
)

# Machine-readable description of score direction and definitions.
data_dictionary = pd.DataFrame([
    ["safe_NORMAL", "Ground-truth NORMAL and predicted NORMAL", "boolean"],
    ["false_NORMAL", "Ground-truth CNV/DME/DRUSEN but predicted NORMAL", "boolean"],
    ["confidence", "Maximum softmax probability; larger means more confident", "0..1"],
    ["entropy", "Predictive entropy using natural logarithms; larger means more uncertain", "0..ln(4)"],
    ["normalized_entropy", "Entropy divided by ln(4)", "0..1"],
    ["classification_margin", "Largest minus second-largest class probability; smaller means more uncertain", "0..1"],
    ["energy_T1", "Negative log-sum-exp of logits at temperature 1; larger is commonly treated as more OOD-like", "real"],
    ["image_id", "Class-relative test image path; stable identifier within this dataset", "text"],
    ["complete_run", f"Epoch >= {INCOMPLETE_EPOCH_THRESHOLD}", "boolean"],
])
data_dictionary.columns = ["field", "definition", "range_or_type"]
data_dictionary.to_csv(audit_dir / "audit_data_dictionary.csv", index=False)

print("\n" + "=" * 80)
print("CLASSIFICATION AUDIT EXPORT COMPLETE")
print("=" * 80)
print(run_summary_df.to_string(index=False))
print("\nComplete-run false-NORMAL decisions:", len(false_normal_df))
print("Unique disease images predicted NORMAL:", false_normal_df["image_id"].nunique())
print(f"\nFiles saved in: {audit_dir}")
for output_file in sorted(audit_dir.glob("*.csv")):
    print(" -", output_file.name)
print("=" * 80)



CLASSIFICATION AUDIT EXPORT COMPLETE
          model  serial  seed  epoch checkpoint_mode                                          checkpoint_filename  complete_run  test_images  accuracy_pct  macro_precision_pct  macro_recall_pct  macro_f1_pct  predicted_NORMAL  safe_NORMAL  false_NORMAL  NORMAL_false_negative_rate_pct
resnet_baseline       1    42     46            best   01_resnet_baseline_seed42_epoch46_best_20260114_001728.pth          True          968     99.070248            99.085542         99.070248     99.070120               241          241             0                        0.000000
      se_resnet       2    42     47            best         02_se_resnet_seed42_epoch47_best_20260114_020922.pth          True          968     98.243802            98.339963         98.243802     98.245417               241          241             0                        0.000000
resnet_baseline       3    84     50            best   03_resnet_baseline_seed84_epoch50_best_20260114_1011

In [15]:
# PERSISTENT HIGH-CONFIDENCE ERROR ANALYSIS
# Objective: identify objective errors that recur across independent architectures/runs.
# Run after COMPLETE CLASSIFICATION-AUDIT EXPORTS. No model inference is repeated.

from collections import Counter
from PIL import Image, ImageDraw, ImageFont

analysis_dir = audit_dir / "persistent_error_analysis"
contact_sheet_dir = analysis_dir / "contact_sheets"
analysis_dir.mkdir(parents=True, exist_ok=True)
contact_sheet_dir.mkdir(parents=True, exist_ok=True)

# Serial 13 stopped at epoch 4 and is already excluded by complete_run.
# Serial 4 duplicates SE-ResNet serial 2 (same seed, predictions and probabilities).
EXCLUDED_DUPLICATE_SERIALS = {4}
HIGH_CONFIDENCE_THRESHOLD = 0.90
CONTROL_RATIO = 1  # Stable correct controls per hard-error image, matched within true class.

independent_df = all_predictions_df[
    all_predictions_df["complete_run"]
    & ~all_predictions_df["serial"].isin(EXCLUDED_DUPLICATE_SERIALS)
].copy()

run_table = (
    independent_df[
        ["model", "serial", "seed", "epoch", "checkpoint_filename"]
    ]
    .drop_duplicates()
    .sort_values(["model", "serial", "seed"])
    .reset_index(drop=True)
)
N_RUNS = len(run_table)
N_ARCHITECTURES = independent_df["model"].nunique()

if N_RUNS == 0:
    raise ValueError("No independent completed evaluations were found.")
if independent_df.groupby("image_id").size().nunique() != 1:
    raise ValueError("Images do not have the same number of independent evaluations.")
if independent_df.groupby("image_id").size().iloc[0] != N_RUNS:
    raise ValueError("At least one image is missing an independent-run prediction.")

independent_df["error"] = ~independent_df["is_correct"].astype(bool)
independent_df["high_confidence_error_90"] = (
    independent_df["error"] & (independent_df["confidence"] >= 0.90)
)
independent_df["high_confidence_error_95"] = (
    independent_df["error"] & (independent_df["confidence"] >= 0.95)
)
independent_df["high_confidence_error_99"] = (
    independent_df["error"] & (independent_df["confidence"] >= 0.99)
)
independent_df["run_key"] = (
    independent_df["model"].astype(str)
    + "_S" + independent_df["serial"].astype(str)
    + "_seed" + independent_df["seed"].astype(str)
)

errors_df = independent_df[independent_df["error"]].copy()
errors_df = errors_df.sort_values(
    ["image_id", "confidence"], ascending=[True, False]
).reset_index(drop=True)

# One row per incorrect model-image decision.
decision_columns = [
    "image_id", "image_path", "true_label", "predicted_label",
    "model", "serial", "seed", "epoch", "run_key",
    "confidence", "entropy", "normalized_entropy",
    "classification_margin", "energy_T1",
    "prob_CNV", "prob_DME", "prob_DRUSEN", "prob_NORMAL",
    "high_confidence_error_90", "high_confidence_error_95",
    "high_confidence_error_99",
]
high_confidence_errors_df = errors_df[
    errors_df["high_confidence_error_90"]
][decision_columns].copy()
high_confidence_errors_df.to_csv(
    analysis_dir / "high_confidence_errors.csv", index=False
)

def joined_unique(values):
    return "|".join(sorted({str(value) for value in values}))

def dominant_wrong_label(values):
    counts = Counter(map(str, values))
    max_count = max(counts.values())
    return "|".join(sorted(label for label, count in counts.items() if count == max_count))

def wrong_label_counts(values):
    counts = Counter(map(str, values))
    return "|".join(f"{label}:{counts[label]}" for label in sorted(counts))

# One row per unique image with at least one error.
persistent_error_summary = (
    errors_df.groupby(["image_id", "image_path", "true_label"], as_index=False)
    .agg(
        error_decisions=("error", "size"),
        architectures_failing=("model", "nunique"),
        failing_architectures=("model", joined_unique),
        dominant_wrong_label=("predicted_label", dominant_wrong_label),
        wrong_label_counts=("predicted_label", wrong_label_counts),
        mean_incorrect_confidence=("confidence", "mean"),
        max_incorrect_confidence=("confidence", "max"),
        min_incorrect_confidence=("confidence", "min"),
        errors_confidence_ge_90=("high_confidence_error_90", "sum"),
        errors_confidence_ge_95=("high_confidence_error_95", "sum"),
        errors_confidence_ge_99=("high_confidence_error_99", "sum"),
        failing_run_keys=("run_key", joined_unique),
    )
)
persistent_error_summary["total_independent_runs"] = N_RUNS
persistent_error_summary["total_architectures"] = N_ARCHITECTURES
persistent_error_summary["error_rate_pct"] = (
    100.0 * persistent_error_summary["error_decisions"] / N_RUNS
)
persistent_error_summary["all_architectures_failed"] = (
    persistent_error_summary["architectures_failing"] == N_ARCHITECTURES
)
persistent_error_summary["persistent_ge_half_runs"] = (
    persistent_error_summary["error_decisions"] >= np.ceil(N_RUNS / 2)
)
persistent_error_summary = persistent_error_summary.sort_values(
    ["error_decisions", "architectures_failing", "max_incorrect_confidence", "image_id"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
persistent_error_summary.insert(
    0, "persistence_rank", np.arange(1, len(persistent_error_summary) + 1)
)
persistent_error_summary.to_csv(
    analysis_dir / "persistent_error_summary.csv", index=False
)

# Architecture-specific error counts and rates for every error image.
architecture_error_counts = (
    independent_df.groupby(["image_id", "model"], as_index=False)
    .agg(
        architecture_runs=("run_key", "nunique"),
        architecture_errors=("error", "sum"),
    )
)
architecture_error_counts["architecture_error_rate_pct"] = (
    100.0
    * architecture_error_counts["architecture_errors"]
    / architecture_error_counts["architecture_runs"]
)

count_wide = architecture_error_counts.pivot(
    index="image_id", columns="model", values="architecture_errors"
).add_prefix("errors_")
rate_wide = architecture_error_counts.pivot(
    index="image_id", columns="model", values="architecture_error_rate_pct"
).add_prefix("error_rate_pct_")
architecture_overlap = (
    persistent_error_summary[
        ["persistence_rank", "image_id", "image_path", "true_label",
         "error_decisions", "architectures_failing", "dominant_wrong_label"]
    ]
    .merge(count_wide.reset_index(), on="image_id", how="left")
    .merge(rate_wide.reset_index(), on="image_id", how="left")
    .sort_values("persistence_rank")
)
architecture_overlap.to_csv(
    analysis_dir / "architecture_overlap.csv", index=False
)

# Image stability table is also used to choose matched, consistently correct controls.
image_stability = (
    independent_df.groupby(["image_id", "image_path", "true_label"], as_index=False)
    .agg(
        correct_decisions=("is_correct", "sum"),
        error_decisions=("error", "sum"),
        mean_true_class_probability=(
            "confidence",
            lambda _: np.nan,  # Replaced below using row-specific true-class probabilities.
        ),
    )
)

probability_columns = {name: f"prob_{name}" for name in CLASS_NAMES}
independent_df["true_class_probability"] = independent_df.apply(
    lambda row: row[probability_columns[row["true_label"]]], axis=1
)
true_probabilities = (
    independent_df.groupby("image_id", as_index=False)["true_class_probability"]
    .mean()
    .rename(columns={"true_class_probability": "mean_true_class_probability_value"})
)
image_stability = image_stability.drop(columns=["mean_true_class_probability"]).merge(
    true_probabilities, on="image_id", how="left"
).rename(columns={
    "mean_true_class_probability_value": "mean_true_class_probability"
})
image_stability["correct_rate_pct"] = (
    100.0 * image_stability["correct_decisions"] / N_RUNS
)

hard_images = persistent_error_summary[
    ["image_id", "image_path", "true_label", "persistence_rank",
     "error_decisions", "error_rate_pct", "architectures_failing",
     "dominant_wrong_label", "max_incorrect_confidence"]
].copy()
hard_images["selection_role"] = "hard_error"
hard_images["selection_reason"] = "At least one error in an independent completed evaluation"

control_frames = []
for true_class, class_hard in hard_images.groupby("true_label"):
    requested = max(1, len(class_hard) * CONTROL_RATIO)
    candidates = image_stability[
        (image_stability["true_label"] == true_class)
        & (image_stability["error_decisions"] == 0)
    ].sort_values(
        ["mean_true_class_probability", "image_id"], ascending=[False, True]
    )
    selected = candidates.head(requested).copy()
    selected["selection_role"] = "matched_correct_control"
    selected["selection_reason"] = (
        f"Correct in all {N_RUNS} independent runs; matched to {true_class}"
    )
    control_frames.append(selected)

controls = (
    pd.concat(control_frames, ignore_index=True)
    if control_frames else pd.DataFrame()
)

manifest_columns = [
    "selection_role", "selection_reason", "image_id", "image_path", "true_label",
    "persistence_rank", "error_decisions", "error_rate_pct",
    "architectures_failing", "dominant_wrong_label", "max_incorrect_confidence",
    "correct_decisions", "correct_rate_pct", "mean_true_class_probability",
]
for column in manifest_columns:
    if column not in hard_images:
        hard_images[column] = np.nan
    if column not in controls:
        controls[column] = np.nan

embedding_manifest = pd.concat(
    [hard_images[manifest_columns], controls[manifest_columns]],
    ignore_index=True,
).sort_values(
    ["selection_role", "true_label", "persistence_rank", "image_id"],
    na_position="last",
).reset_index(drop=True)
embedding_manifest.to_csv(
    analysis_dir / "embedding_extraction_manifest.csv", index=False
)

def make_contact_sheet(rows, output_path, title, max_images=40, columns=5):
    rows = rows.head(max_images).copy()
    if rows.empty:
        print(f"Skipped empty contact sheet: {title}")
        return

    thumb_w, thumb_h = 224, 224
    label_h, title_h, gap = 70, 44, 12
    ncols = min(columns, len(rows))
    nrows = int(np.ceil(len(rows) / ncols))
    canvas_w = gap + ncols * (thumb_w + gap)
    canvas_h = title_h + gap + nrows * (thumb_h + label_h + gap)
    canvas = Image.new("RGB", (canvas_w, canvas_h), "white")
    draw = ImageDraw.Draw(canvas)
    font = ImageFont.load_default()
    draw.text((gap, 12), title, fill="black", font=font)

    for position, (_, row) in enumerate(rows.iterrows()):
        col = position % ncols
        grid_row = position // ncols
        x = gap + col * (thumb_w + gap)
        y = title_h + gap + grid_row * (thumb_h + label_h + gap)
        try:
            with Image.open(row["image_path"]) as source:
                image = source.convert("RGB")
                image.thumbnail((thumb_w, thumb_h))
                tile = Image.new("RGB", (thumb_w, thumb_h), "black")
                tile.paste(
                    image,
                    ((thumb_w - image.width) // 2, (thumb_h - image.height) // 2),
                )
                canvas.paste(tile, (x, y))
        except Exception as exc:
            draw.rectangle((x, y, x + thumb_w, y + thumb_h), outline="red", width=3)
            draw.text((x + 5, y + 5), f"IMAGE ERROR\n{type(exc).__name__}", fill="red", font=font)

        rank = row.get("persistence_rank", "")
        errors = row.get("error_decisions", "")
        architectures = row.get("architectures_failing", "")
        predicted = row.get("dominant_wrong_label", row.get("predicted_label", ""))
        confidence = row.get("max_incorrect_confidence", row.get("confidence", np.nan))
        confidence_text = f"{float(confidence):.3f}" if pd.notna(confidence) else "NA"
        label = (
            f"Rank {rank} | {row['true_label']} -> {predicted}\n"
            f"Errors {errors}/{N_RUNS} | Arch {architectures}/{N_ARCHITECTURES}\n"
            f"Max wrong confidence {confidence_text}"
        )
        draw.multiline_text((x, y + thumb_h + 4), label, fill="black", font=font, spacing=2)

    canvas.save(output_path, dpi=(200, 200))
    print("Saved contact sheet:", output_path.name)

make_contact_sheet(
    persistent_error_summary,
    contact_sheet_dir / "top_persistent_errors.png",
    f"Top persistent errors ({N_RUNS} independent runs)",
)
make_contact_sheet(
    persistent_error_summary[
        persistent_error_summary["all_architectures_failed"]
    ],
    contact_sheet_dir / "errors_shared_by_all_architectures.png",
    "Errors observed in all architectures",
)
make_contact_sheet(
    persistent_error_summary[
        persistent_error_summary["errors_confidence_ge_90"] > 0
    ],
    contact_sheet_dir / "persistent_high_confidence_errors.png",
    "Persistent images with at least one >=90% confidence error",
)
make_contact_sheet(
    persistent_error_summary[
        persistent_error_summary["true_label"] == "NORMAL"
    ],
    contact_sheet_dir / "normal_to_disease_errors.png",
    "Ground-truth NORMAL images predicted as disease",
)

analysis_dictionary = pd.DataFrame([
    ["independent completed run", "complete_run is true and duplicate serial 4 is excluded"],
    ["persistent error", "an image misclassified by at least one independent completed run"],
    ["all_architectures_failed", f"at least one error occurred in each of {N_ARCHITECTURES} architectures"],
    ["high-confidence error", f"incorrect prediction with confidence >= {HIGH_CONFIDENCE_THRESHOLD:.2f}"],
    ["matched correct control", f"same true class and correct in all {N_RUNS} independent runs"],
    ["embedding manifest", "hard-error images plus deterministic stable class-matched controls"],
], columns=["term", "definition"])
analysis_dictionary.to_csv(
    analysis_dir / "persistent_error_data_dictionary.csv", index=False
)

print("\n" + "=" * 80)
print("PERSISTENT-ERROR ANALYSIS COMPLETE")
print("=" * 80)
print(f"Independent completed evaluations: {N_RUNS}")
print(f"Architectures: {N_ARCHITECTURES}")
print(f"Incorrect decisions: {len(errors_df)}")
print(f"Unique images with >=1 error: {len(persistent_error_summary)}")
print(
    "Images failed by all architectures:",
    int(persistent_error_summary["all_architectures_failed"].sum()),
)
print(f"Decision-level errors at >=90% confidence: {len(high_confidence_errors_df)}")
print(f"Embedding manifest rows: {len(embedding_manifest)}")
print(f"\nFiles saved in: {analysis_dir}")
for output_file in sorted(analysis_dir.rglob("*")):
    if output_file.is_file():
        print(" -", output_file.relative_to(analysis_dir))
print("=" * 80)


Saved contact sheet: top_persistent_errors.png
Saved contact sheet: errors_shared_by_all_architectures.png
Saved contact sheet: persistent_high_confidence_errors.png
Saved contact sheet: normal_to_disease_errors.png

PERSISTENT-ERROR ANALYSIS COMPLETE
Independent completed evaluations: 16
Architectures: 4
Incorrect decisions: 232
Unique images with >=1 error: 69
Images failed by all architectures: 7
Decision-level errors at >=90% confidence: 65
Embedding manifest rows: 138

Files saved in: D:\Keel\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints\evaluation_results\classification_audit\persistent_error_analysis
 - architecture_overlap.csv
 - contact_sheets\errors_shared_by_all_architectures.png
 - contact_sheets\normal_to_disease_errors.png
 - contact_sheets\persistent_high_confidence_errors.png
 - contact_sheets\top_persistent_errors.png
 - embedding_extraction_manifest.csv
 - high_confidence_errors.csv
 - persistent_error_data_dictionary.csv
 - persistent_e

In [16]:
def get_embedding_hook_target(model, model_type):
    if model_type in ('resnet_baseline', 'se_resnet', 'cbam_resnet'):
        return model.avgpool, 'forward'
    elif model_type == 'vit_b16':
        return model.heads, 'pre'
    raise ValueError(f"Unknown model type: {model_type}")

def extract_embeddings(model, model_type, loader, device):
    model.eval()
    store = []
    hook_module, hook_kind = get_embedding_hook_target(model, model_type)

    def fwd_hook(module, inp, out):
        store.append(torch.flatten(out, 1).detach().cpu().numpy())
    def pre_hook(module, inp):
        store.append(inp[0].detach().cpu().numpy())

    handle = (hook_module.register_forward_hook(fwd_hook) if hook_kind == 'forward'
              else hook_module.register_forward_pre_hook(pre_hook))

    with torch.no_grad():
        for images, _ in tqdm(loader, desc="Extracting embeddings", leave=False):
            images = images.to(device, non_blocking=True)
            model(images)

    handle.remove()
    return np.concatenate(store, axis=0)

In [17]:
TRAIN_PATH = ROOT / "Data_Kermany_OCT2017" / "train"

full_train_dataset = ImageFolder(root=str(TRAIN_PATH), transform=test_transform)  # deterministic, matches test preprocessing
full_train_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
train_labels_array = np.array([label for _, label in full_train_dataset.samples])

normal_idx = CLASS_NAMES.index("NORMAL")
print(f"Training set: {len(full_train_dataset)} images, {int((train_labels_array == normal_idx).sum())} NORMAL")

Training set: 55792 images, 22677 NORMAL


In [18]:
from sklearn.covariance import LedoitWolf

geometry_rows = []

for r in results:
    if r["epoch"] < INCOMPLETE_EPOCH_THRESHOLD:
        continue  # same exclusion rule as the audit exports

    serial, seed, model_name = r["serial"], r["seed"], r["model_name"]
    print(f"\n{'='*80}\n{model_name} | serial {serial} | seed {seed}\n{'='*80}")

    model = create_model(model_name, NUM_CLASSES)
    ckpt_path = CHECKPOINT_DIR / r["checkpoint_filename"]
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(DEVICE)

    # Fit reference geometry on the TRAINING set only
    train_embeddings = extract_embeddings(model, model_name, full_train_loader, DEVICE)
    normal_train_emb = train_embeddings[train_labels_array == normal_idx]
    normal_mean = normal_train_emb.mean(axis=0)
    inv_cov = LedoitWolf().fit(normal_train_emb).precision_

    disease_directions = {}
    for cls_name in CLASS_NAMES:
        if cls_name == "NORMAL":
            continue
        cls_mean = train_embeddings[train_labels_array == CLASS_NAMES.index(cls_name)].mean(axis=0)
        d = cls_mean - normal_mean
        disease_directions[cls_name] = d / np.linalg.norm(d)

    # Score the TEST set against this model's own reference geometry
    test_embeddings = extract_embeddings(model, model_name, test_loader, DEVICE)
    diffs = test_embeddings - normal_mean
    mahal_dist = np.sqrt(np.maximum(np.einsum('ij,jk,ik->i', diffs, inv_cov, diffs), 0))

    diff_norms = np.linalg.norm(diffs, axis=1, keepdims=True)
    diff_norms[diff_norms == 0] = 1e-12
    unit_diffs = diffs / diff_norms

    frame = pd.DataFrame({
        "image_id": relative_paths,   # from cell 14 — same order as test_loader
        "model": model_name, "serial": serial, "seed": seed,
        "mahalanobis_distance": mahal_dist,
    })
    for cls_name, direction in disease_directions.items():
        frame[f"cosine_{cls_name}"] = unit_diffs @ direction

    geometry_rows.append(frame)
    del model, train_embeddings, test_embeddings
    torch.cuda.empty_cache()

geometry_df = pd.concat(geometry_rows, ignore_index=True)
geometry_dir = audit_dir / "geometry"
geometry_dir.mkdir(parents=True, exist_ok=True)
geometry_df.to_csv(geometry_dir / "mahalanobis_cosine_all_runs.csv", index=False)
print(f"\nSaved {len(geometry_df)} rows to {geometry_dir / 'mahalanobis_cosine_all_runs.csv'}")


resnet_baseline | serial 1 | seed 42



se_resnet | serial 2 | seed 42



resnet_baseline | serial 3 | seed 84



se_resnet | serial 4 | seed 42



se_resnet | serial 5 | seed 84


...


vit_b16 | serial 15 | seed 126



vit_b16 | serial 16 | seed 3407



resnet_baseline | serial 17 | seed 3407



cbam_resnet | serial 18 | seed 3407



Saved 16456 rows to D:\Keel\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints\evaluation_results\classification_audit\geometry\mahalanobis_cosine_all_runs.csv


In [19]:
normal_predicted_df = all_predictions_df[
    all_predictions_df["complete_run"] & all_predictions_df["predicted_NORMAL"]
][["image_id", "model", "serial", "seed"]]

geo_normal_df = geometry_df.merge(normal_predicted_df, on=["image_id","model","serial","seed"], how="inner")
geo_normal_df["mahalanobis_percentile"] = (
    geo_normal_df.groupby(["model","serial","seed"])["mahalanobis_distance"].rank(pct=True) * 100
)
geo_normal_df.to_csv(geometry_dir / "normal_predicted_geometry_ranked.csv", index=False)

In [20]:
from scipy.stats import spearmanr
from itertools import combinations

geo_normal_df["run_key"] = (geo_normal_df["model"] + "_S" + geo_normal_df["serial"].astype(str)
                             + "_seed" + geo_normal_df["seed"].astype(str))
wide = geo_normal_df.pivot_table(index="image_id", columns="run_key", values="mahalanobis_percentile")

pair_rows = []
for a, b in combinations(wide.columns, 2):
    common = wide[[a, b]].dropna()
    if len(common) < 5:
        continue
    rho, _ = spearmanr(common[a], common[b])
    top_a, top_b = set(common.index[common[a] >= 90]), set(common.index[common[b] >= 90])
    jacc = len(top_a & top_b) / len(top_a | top_b) if (top_a | top_b) else np.nan
    pair_rows.append({"run_a": a, "run_b": b, "n_common": len(common),
                       "spearman_rho": rho, "jaccard_top10pct": jacc,
                       "same_architecture": a.split("_S")[0] == b.split("_S")[0]})

pairwise_df = pd.DataFrame(pair_rows).sort_values("spearman_rho", ascending=False)
pairwise_df.to_csv(geometry_dir / "pairwise_reproducibility_metrics.csv", index=False)

print("Same-architecture (seed-level) pairs — mean Spearman:",
      pairwise_df[pairwise_df.same_architecture]["spearman_rho"].mean())
print("Cross-architecture pairs — mean Spearman:",
      pairwise_df[~pairwise_df.same_architecture]["spearman_rho"].mean())
print("\nFull table:")
print(pairwise_df.to_string(index=False))

Same-architecture (seed-level) pairs — mean Spearman: 0.3171270980038722
Cross-architecture pairs — mean Spearman: 0.10788873080410902

Full table:
                       run_a                        run_b  n_common  spearman_rho  jaccard_top10pct  same_architecture
         se_resnet_S2_seed42          se_resnet_S4_seed42       241      1.000000          1.000000               True
         vit_b16_S15_seed126         vit_b16_S16_seed3407       242      0.600570          0.219512               True
          vit_b16_S12_seed42          vit_b16_S15_seed126       242      0.547584          0.282051               True
          vit_b16_S12_seed42         vit_b16_S16_seed3407       242      0.494851          0.219512               True
          vit_b16_S14_seed84          vit_b16_S15_seed126       242      0.458412          0.250000               True
          vit_b16_S12_seed42           vit_b16_S14_seed84       242      0.449325          0.190476               True
          vit_b16_S

In [21]:
# Cell 21 — load RETFound and verify class order (run this alone first, do not proceed until confirmed):

import timm
from huggingface_hub import hf_hub_download
import json as _json

from torchvision import transforms as T

retfound_transform_nocrop = T.Compose([
    T.Resize((224, 224), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

RETFOUND_MODEL_ID = "hf_hub:bitfount/RETFound_MAE_OCT_CNV_DME_DRU"

retfound_model = timm.create_model(RETFOUND_MODEL_ID, pretrained=True)
retfound_model.eval().to(DEVICE)

# Confirm class order against CLASS_NAMES = ['CNV','DME','DRUSEN','NORMAL']
try:
    cfg_path = hf_hub_download(repo_id="bitfount/RETFound_MAE_OCT_CNV_DME_DRU", filename="config.json")
    with open(cfg_path) as f:
        retfound_cfg = _json.load(f)
    print("RETFound config.json:", _json.dumps(retfound_cfg, indent=2))
except Exception as e:
    print("Could not fetch config.json:", e)

print("\nRETFound pretrained_cfg:", getattr(retfound_model, 'pretrained_cfg', None))
print("Model output dim (num_classes):", retfound_model.num_classes if hasattr(retfound_model, 'num_classes') else "check head shape")

# Get correct preprocessing for this model — do not reuse test_transform from your own models
retfound_data_cfg = timm.data.resolve_data_config({}, model=retfound_model)
retfound_transform = timm.data.create_transform(**retfound_data_cfg)
print("\nRETFound expected preprocessing:", retfound_data_cfg)

RETFound config.json: {
  "architecture": "vit_large_patch16_224",
  "num_classes": 4,
  "num_features": 1024,
  "global_pool": "token",
  "pretrained_cfg": {
    "tag": "augreg_in21k_ft_in1k",
    "custom_load": true,
    "input_size": [
      3,
      224,
      224
    ],
    "fixed_input_size": true,
    "interpolation": "bicubic",
    "crop_pct": 0.9,
    "crop_mode": "center",
    "mean": [
      0.485,
      0.456,
      0.406
    ],
    "std": [
      0.229,
      0.224,
      0.225
    ],
    "num_classes": 1000,
    "pool_size": null,
    "first_conv": "patch_embed.proj",
    "classifier": "head"
  }
}

RETFound pretrained_cfg: {'hf_hub_id': 'bitfount/RETFound_MAE_OCT_CNV_DME_DRU', 'source': 'hf-hub', 'architecture': 'vit_large_patch16_224', 'tag': 'augreg_in21k_ft_in1k', 'custom_load': True, 'input_size': [3, 224, 224], 'fixed_input_size': True, 'interpolation': 'bicubic', 'crop_pct': 0.9, 'crop_mode': 'center', 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225], 'n

In [22]:
# Cell 22 — RETFound-specific data loaders (its own preprocessing, not your models'):

retfound_train_dataset = ImageFolder(root=str(TRAIN_PATH), transform=retfound_transform_nocrop)
retfound_train_loader = DataLoader(retfound_train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
retfound_train_labels = np.array([label for _, label in retfound_train_dataset.samples])

retfound_test_dataset = ImageFolder(root=str(TEST_PATH), transform=retfound_transform_nocrop)
retfound_test_loader = DataLoader(retfound_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [23]:
# Cell 23 — RETFound embeddings + geometry (its own fixed reference frame):

import os
def extract_retfound_embeddings(model, loader, device):
    model.eval()
    store = []
    with torch.no_grad():
        for images, _ in tqdm(loader, desc="RETFound embeddings", leave=False):
            images = images.to(device, non_blocking=True)
            feats = model.forward_features(images)   # penultimate representation
            if feats.ndim == 3:      # (B, tokens, dim) -> CLS token
                feats = feats[:, 0]
            store.append(feats.detach().cpu().numpy())
    return np.concatenate(store, axis=0)

retfound_train_emb = extract_retfound_embeddings(retfound_model, retfound_train_loader, DEVICE)
retfound_test_emb  = extract_retfound_embeddings(retfound_model, retfound_test_loader, DEVICE)

normal_idx = CLASS_NAMES.index("NORMAL")
normal_train_emb = retfound_train_emb[retfound_train_labels == normal_idx]
normal_mean = normal_train_emb.mean(axis=0)
inv_cov = LedoitWolf().fit(normal_train_emb).precision_

diffs = retfound_test_emb - normal_mean
retfound_mahal = np.sqrt(np.maximum(np.einsum('ij,jk,ik->i', diffs, inv_cov, diffs), 0))

retfound_geo = pd.DataFrame({
    #"image_id": [os.path.relpath(p, TEST_PATH) for p,_ in retfound_test_dataset.samples],
    "image_id": [str(Path(p).relative_to(TEST_PATH)).replace("\\", "/") for p, _ in retfound_test_dataset.samples],
    "retfound_mahalanobis": retfound_mahal
})
retfound_geo["retfound_percentile"] = retfound_geo["retfound_mahalanobis"].rank(pct=True) * 100
retfound_geo.to_csv(geometry_dir / "retfound_geometry.csv", index=False)
print(f"Saved {len(retfound_geo)} rows.")

Saved 968 rows.


In [24]:
# Cell 24 — compare RETFound's ranking against each of your four architectures (bonus evidence, not the primary criterion):

own_wide = geo_normal_df.pivot_table(index="image_id", columns="model", values="mahalanobis_percentile", aggfunc="mean")
merged = own_wide.merge(retfound_geo.set_index("image_id")[["retfound_percentile"]], left_index=True, right_index=True, how="inner")

for arch in own_wide.columns:
    common = merged[[arch, "retfound_percentile"]].dropna()
    rho, _ = spearmanr(common[arch], common["retfound_percentile"])
    print(f"RETFound vs {arch}: Spearman rho = {rho:.3f}  (n={len(common)})")

RETFound vs cbam_resnet: Spearman rho = 0.141  (n=242)
RETFound vs resnet_baseline: Spearman rho = 0.047  (n=241)
RETFound vs se_resnet: Spearman rho = 0.010  (n=242)
RETFound vs vit_b16: Spearman rho = 0.253  (n=242)


In [25]:
# Cell 25 — empirically verify RETFound's class order before doing anything else:

from itertools import permutations
from torch.utils.data import Subset

# Use a moderate, balanced sample from the training set (fast, doesn't need the full set)
verify_dataset = ImageFolder(root=str(TRAIN_PATH), transform=retfound_transform_nocrop)
verify_labels = np.array([label for _, label in verify_dataset.samples])

# Balanced subsample: ~100 per class is enough to distinguish orderings clearly
rng = np.random.default_rng(42)
sample_idx = []
for c in range(len(CLASS_NAMES)):
    class_idx = np.where(verify_labels == c)[0]
    sample_idx.extend(rng.choice(class_idx, size=min(100, len(class_idx)), replace=False))
sample_idx = np.array(sample_idx)

verify_subset = Subset(verify_dataset, sample_idx)
verify_loader = DataLoader(verify_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
verify_true = verify_labels[sample_idx]

retfound_model.eval()
verify_preds = []
with torch.no_grad():
    for images, _ in tqdm(verify_loader, desc="Verifying class order"):
        images = images.to(DEVICE)
        outputs = retfound_model(images)
        verify_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
verify_preds = np.array(verify_preds)

# Try every permutation of the 4-class label order and score accuracy under each
print("Testing all permutations of class order against ground truth:\n")
best_acc, best_perm = 0, None
for perm in permutations(range(4)):
    remapped_preds = np.array([perm[p] for p in verify_preds])
    acc = (remapped_preds == verify_true).mean()
    label_str = [CLASS_NAMES[i] for i in perm]
    print(f"  If RETFound's [0,1,2,3] maps to {label_str}: accuracy = {acc:.3f}")
    if acc > best_acc:
        best_acc, best_perm = acc, perm

print(f"\nBest match: RETFound's output order maps to {[CLASS_NAMES[i] for i in best_perm]}")
print(f"Accuracy under this mapping: {best_acc:.3f}")

if best_acc < 0.85:
    print("\n⚠️ WARNING: best-fit accuracy is low. Class order could not be confidently verified — do not proceed until this is resolved.")

Verifying class order: 100%|███████████████████████████████████████████████████████████| 13/13 [00:03<00:00,  4.01it/s]

Testing all permutations of class order against ground truth:

  If RETFound's [0,1,2,3] maps to ['CNV', 'DME', 'DRUSEN', 'NORMAL']: accuracy = 0.818
  If RETFound's [0,1,2,3] maps to ['CNV', 'DME', 'NORMAL', 'DRUSEN']: accuracy = 0.482
  If RETFound's [0,1,2,3] maps to ['CNV', 'DRUSEN', 'DME', 'NORMAL']: accuracy = 0.507
  If RETFound's [0,1,2,3] maps to ['CNV', 'DRUSEN', 'NORMAL', 'DME']: accuracy = 0.280
  If RETFound's [0,1,2,3] maps to ['CNV', 'NORMAL', 'DME', 'DRUSEN']: accuracy = 0.300
  If RETFound's [0,1,2,3] maps to ['CNV', 'NORMAL', 'DRUSEN', 'DME']: accuracy = 0.407
  If RETFound's [0,1,2,3] maps to ['DME', 'CNV', 'DRUSEN', 'NORMAL']: accuracy = 0.425
  If RETFound's [0,1,2,3] maps to ['DME', 'CNV', 'NORMAL', 'DRUSEN']: accuracy = 0.090
  If RETFound's [0,1,2,3] maps to ['DME', 'DRUSEN', 'CNV', 'NORMAL']: accuracy = 0.290
  If RETFound's [0,1,2,3] maps to ['DME', 'DRUSEN', 'NORMAL', 'CNV']: accuracy = 0.050
  If RETFound's [0,1,2,3] maps to ['DME', 'NORMAL', 'CNV', 'DRUSEN'

In [26]:
# Cell 26 — confirm layer names before hooking anything:
# Print named modules for one instance of each architecture type, to confirm hook targets
for arch_name in ['resnet_baseline', 'se_resnet', 'cbam_resnet']:
    m = create_model(arch_name, NUM_CLASSES)
    print(f"\n{'='*60}\n{arch_name}\n{'='*60}")
    for name, _ in m.named_children():
        print(" ", name)
    del m

print(f"\n{'='*60}\nvit_b16 (torchvision)\n{'='*60}")
m = create_model('vit_b16', NUM_CLASSES)
print("Top-level:", [n for n, _ in m.named_children()])
print("Number of encoder blocks:", len(m.encoder.layers))
del m


resnet_baseline
  conv1
  bn1
  relu
  maxpool
  layer1
  layer2
  layer3
  layer4
  avgpool
  fc

se_resnet
  conv1
  bn1
  relu
  maxpool
  layer1
  layer2
  layer3
  layer4
  se1
  se2
  se3
  se4
  avgpool
  fc

cbam_resnet
  conv1
  bn1
  relu
  maxpool
  layer1
  layer2
  layer3
  layer4
  cbam1
  cbam2
  cbam3
  cbam4
  avgpool
  fc

vit_b16 (torchvision)
Top-level: ['conv_proj', 'encoder', 'heads']
Number of encoder blocks: 12


In [29]:
# Cell 27 — multi-layer embedding extraction (relative-depth matched across architectures)
#
# Multi-depth protocol motivated by:
#   Lee, K., Lee, K., Lee, H., & Shin, J. (2018). A simple unified framework for
#   detecting out-of-distribution samples and adversarial attacks. NeurIPS 31.
#   — original Mahalanobis-OOD method combined scores from multiple network layers,
#     not a single (e.g. penultimate) layer.
#   Anthony, H., & Kamnitsas, K. (2023). On the use of Mahalanobis distance for
#   out-of-distribution detection with neural networks for medical imaging.
#   UNSURE Workshop, MICCAI 2023, Springer LNCS.
#   — showed no single network depth is universally optimal for medical imaging;
#     optimal depth depends on the type of atypicality being detected.


def get_multilayer_hook_targets(model, model_type):
    """
    Returns {layer_name: (module, hook_kind)} for the three prespecified
    relative-depth stages: mid, late, final.

    For se_resnet/cbam_resnet, the attention blocks (se1-4, cbam1-4) are
    SEPARATE sibling modules that run AFTER each layer stage, not nested
    inside it (confirmed via named_children() inspection). Hooking se2/cbam2
    (post-attention) rather than layer2 (pre-attention) is required so the
    'mid'/'late' representations reflect what actually flows forward into
    the next stage — matching what resnet_baseline's layer2/layer3 hooks
    naturally capture, since resnet_baseline has no attention step in between.
    """
    if model_type == 'resnet_baseline':
        return {
            'mid':   (model.layer2, 'forward'),
            'late':  (model.layer3, 'forward'),
            'final': (model.avgpool, 'forward'),
        }
    elif model_type == 'se_resnet':
        return {
            'mid':   (model.se2, 'forward'),    # post-attention, matches what flows into layer3
            'late':  (model.se3, 'forward'),
            'final': (model.avgpool, 'forward'),
        }
    elif model_type == 'cbam_resnet':
        return {
            'mid':   (model.cbam2, 'forward'),
            'late':  (model.cbam3, 'forward'),
            'final': (model.avgpool, 'forward'),
        }
    elif model_type == 'vit_b16':
        return {
            'mid':   (model.encoder.layers[4], 'forward'),
            'late':  (model.encoder.layers[8], 'forward'),
            'final': (model.heads, 'pre'),
        }
    raise ValueError(f"Unknown model type: {model_type}")


def extract_multilayer_embeddings(model, model_type, loader, device):
    """
    Extracts pooled embeddings at three relative-depth stages (mid, late, final)
    in a single forward pass per image — no need to re-run inference per layer.

    Handles two output shapes generically:
      - 4D CNN feature maps (B, C, H, W) -> global average pooled to (B, C)
      - 3D ViT token sequences (B, tokens, dim) -> CLS token (B, dim)
    If a hooked module returns a tuple (some custom blocks do), the first
    element is used.
    """
    model.eval()
    targets = get_multilayer_hook_targets(model, model_type)
    stores = {name: [] for name in targets}
    handles = []

    def make_fwd_hook(name):
        def hook(module, inp, out):
            feat = out[0] if isinstance(out, tuple) else out
            if feat.ndim == 4:                      # CNN feature map (B,C,H,W)
                feat = torch.nn.functional.adaptive_avg_pool2d(feat, 1).flatten(1)
            elif feat.ndim == 3:                     # ViT block output (B,tokens,dim)
                feat = feat[:, 0]                    # CLS token
            else:
                feat = torch.flatten(feat, 1)
            stores[name].append(feat.detach().cpu().numpy())
        return hook

    def make_pre_hook(name):
        def hook(module, inp):
            feat = inp[0]
            stores[name].append(feat.detach().cpu().numpy())
        return hook

    for name, (module, kind) in targets.items():
        if kind == 'forward':
            handles.append(module.register_forward_hook(make_fwd_hook(name)))
        else:
            handles.append(module.register_forward_pre_hook(make_pre_hook(name)))

    with torch.no_grad():
        for images, _ in tqdm(loader, desc=f"Multi-layer extraction ({model_type})", leave=False):
            images = images.to(device, non_blocking=True)
            model(images)

    for h in handles:
        h.remove()

    return {name: np.concatenate(arrs, axis=0) for name, arrs in stores.items()}
    

In [33]:
from torch.utils.data import TensorDataset

test_model = create_model('se_resnet', NUM_CLASSES).to(DEVICE)
one_batch_images, _ = next(iter(test_loader))
one_batch_images = one_batch_images.to(DEVICE)
result = extract_multilayer_embeddings(test_model, 'se_resnet', DataLoader(TensorDataset(one_batch_images, torch.zeros(len(one_batch_images))), batch_size=len(one_batch_images)), DEVICE)
for k, v in result.items():
    print(k, v.shape)
del test_model

mid (32, 512)
late (32, 1024)
final (32, 2048)


In [39]:

# Cell 28 — per-layer geometry (Mahalanobis + disease-direction cosine), per-architecture,
# over your four models. Depths tested per Lee et al. (2018) / Anthony & Kamnitsas (2023) —
# see Cell 27 for full citations. All three depths (mid/late/final) are reported regardless
# of outcome — no post-hoc best-layer selection.

multilayer_rows = []

for r in results:
    if r["epoch"] < INCOMPLETE_EPOCH_THRESHOLD:
        continue
    if not r.get("checkpoint_filename"):
        continue

    serial, seed, model_name = r["serial"], r["seed"], r["model_name"]
    print(f"\n{model_name} | serial {serial} | seed {seed}")

    model = create_model(model_name, NUM_CLASSES)
    ckpt = torch.load(CHECKPOINT_DIR / r["checkpoint_filename"], map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(DEVICE)

    train_multi = extract_multilayer_embeddings(model, model_name, full_train_loader, DEVICE)
    test_multi  = extract_multilayer_embeddings(model, model_name, test_loader, DEVICE)

    for layer_name in train_multi:
        normal_train = train_multi[layer_name][train_labels_array == normal_idx]
        mean_vec = normal_train.mean(axis=0)
        inv_cov = LedoitWolf().fit(normal_train).precision_

        disease_directions = {}
        for cls_name in CLASS_NAMES:
            if cls_name == "NORMAL":
                continue
            cls_train = train_multi[layer_name][train_labels_array == CLASS_NAMES.index(cls_name)]
            d = cls_train.mean(axis=0) - mean_vec
            disease_directions[cls_name] = d / np.linalg.norm(d)

        diffs = test_multi[layer_name] - mean_vec
        mahal = np.sqrt(np.maximum(np.einsum('ij,jk,ik->i', diffs, inv_cov, diffs), 0))

        diff_norms = np.linalg.norm(diffs, axis=1, keepdims=True)
        diff_norms[diff_norms == 0] = 1e-12
        unit_diffs = diffs / diff_norms

        depth_map = {'mid': 0.33, 'late': 0.67, 'final': 1.0}
        frame = pd.DataFrame({
            "image_id": relative_paths, "model": model_name, "serial": serial, "seed": seed,
            "layer_name": layer_name, "relative_depth": depth_map[layer_name],
            "mahalanobis_distance": mahal,
        })
        for cls_name, direction in disease_directions.items():
            frame[f"cosine_{cls_name}"] = unit_diffs @ direction

        multilayer_rows.append(frame)

    del model, train_multi, test_multi
    torch.cuda.empty_cache()

multilayer_df = pd.concat(multilayer_rows, ignore_index=True)
multilayer_df.to_csv(geometry_dir / "multilayer_geometry_all_runs.csv", index=False)
print(f"\nSaved {len(multilayer_df)} rows.")


resnet_baseline | serial 1 | seed 42



se_resnet | serial 2 | seed 42



resnet_baseline | serial 3 | seed 84



se_resnet | serial 4 | seed 42



se_resnet | serial 5 | seed 84


...


vit_b16 | serial 15 | seed 126



vit_b16 | serial 16 | seed 3407



resnet_baseline | serial 17 | seed 3407



cbam_resnet | serial 18 | seed 3407



Saved 49368 rows.


In [40]:
print("RETFound top-level modules:")
for name, _ in retfound_model.named_children():
    print(" ", name)

# timm ViT models typically expose blocks via .blocks, not .encoder.layers (that's torchvision's convention)
if hasattr(retfound_model, 'blocks'):
    print(f"\nNumber of transformer blocks: {len(retfound_model.blocks)}")
else:
    print("\nNo '.blocks' attribute found — structure differs from standard timm ViT, inspect named_children() above")

RETFound top-level modules:
  patch_embed
  pos_drop
  patch_drop
  norm_pre
  blocks
  norm
  fc_norm
  head_drop
  head

Number of transformer blocks: 24


In [41]:

# Cell 30 — RETFound multi-layer embedding extraction, matched relative depths.
# Same multi-depth rationale as Cell 27 (Lee et al., 2018; Anthony & Kamnitsas, 2023),
# applied here to an independently pretrained model (Zhou et al., 2023 — RETFound)
# to test whether the depth-dependence finding holds beyond the four in-house architectures.

def extract_retfound_multilayer_embeddings(model, loader, device, mid_block_idx, late_block_idx):
    """
    Extracts RETFound CLS-token embeddings at three matched relative depths:
    mid, late (via forward hooks on the specified transformer blocks),
    and final (via model.forward_features(), consistent with cell 23's approach).
    """
    model.eval()
    stores = {'mid': [], 'late': [], 'final': []}

    def make_block_hook(name):
        def hook(module, inp, out):
            feat = out[0] if isinstance(out, tuple) else out
            stores[name].append(feat[:, 0].detach().cpu().numpy())  # CLS token
        return hook

    h_mid = model.blocks[mid_block_idx].register_forward_hook(make_block_hook('mid'))
    h_late = model.blocks[late_block_idx].register_forward_hook(make_block_hook('late'))

    with torch.no_grad():
        for images, _ in tqdm(loader, desc="RETFound multi-layer extraction", leave=False):
            images = images.to(device, non_blocking=True)
            feats = model.forward_features(images)
            if feats.ndim == 3:
                feats = feats[:, 0]
            stores['final'].append(feats.detach().cpu().numpy())

    h_mid.remove()
    h_late.remove()
    return {name: np.concatenate(arrs, axis=0) for name, arrs in stores.items()}

In [42]:
# Cell 31 — compute geometry (Mahalanobis + disease-direction cosine) at all three RETFound
# depths, using its own train-set NORMAL reference. Same protocol as Cell 28 — see Cell 27
# for full citations (Lee et al., 2018; Anthony & Kamnitsas, 2023).

mid_idx, late_idx = 8, 16   # confirmed against cell 29's actual block count (24 blocks total)

retfound_train_multi = extract_retfound_multilayer_embeddings(retfound_model, retfound_train_loader, DEVICE, mid_idx, late_idx)
retfound_test_multi  = extract_retfound_multilayer_embeddings(retfound_model, retfound_test_loader, DEVICE, mid_idx, late_idx)

retfound_multilayer_rows = []
depth_map = {'mid': 0.33, 'late': 0.67, 'final': 1.0}

for layer_name in ['mid', 'late', 'final']:
    normal_train = retfound_train_multi[layer_name][retfound_train_labels == normal_idx]
    mean_vec = normal_train.mean(axis=0)
    inv_cov = LedoitWolf().fit(normal_train).precision_

    disease_directions = {}
    for cls_name in CLASS_NAMES:
        if cls_name == "NORMAL":
            continue
        cls_train = retfound_train_multi[layer_name][retfound_train_labels == CLASS_NAMES.index(cls_name)]
        d = cls_train.mean(axis=0) - mean_vec
        d_norm = np.linalg.norm(d)

        if d_norm < 1e-12:
            raise ValueError(
                f"Near-zero disease direction for {cls_name} at layer {layer_name}"
            )

        disease_directions[cls_name] = d / d_norm

    diffs = retfound_test_multi[layer_name] - mean_vec
    mahal = np.sqrt(np.maximum(np.einsum('ij,jk,ik->i', diffs, inv_cov, diffs), 0))

    diff_norms = np.linalg.norm(diffs, axis=1, keepdims=True)
    diff_norms[diff_norms == 0] = 1e-12
    unit_diffs = diffs / diff_norms

    frame = pd.DataFrame({
        "image_id": [
            str(Path(p).relative_to(TEST_PATH)).replace("\\", "/")
            for p, _ in retfound_test_dataset.samples
        ],
        "model": "retfound",
        "serial": 0,
        "seed": 0,
        "layer_name": layer_name,
        "relative_depth": depth_map[layer_name],
        "mahalanobis_distance": mahal,
    })
    for cls_name, direction in disease_directions.items():
        frame[f"cosine_{cls_name}"] = unit_diffs @ direction

    retfound_multilayer_rows.append(frame)

retfound_multilayer_df = pd.concat(retfound_multilayer_rows, ignore_index=True)
retfound_multilayer_df.to_csv(geometry_dir / "retfound_multilayer_geometry.csv", index=False)
print(f"Saved {len(retfound_multilayer_df)} rows.")

Saved 2904 rows.
